## Pose Estimation — Rostro · Manos · Cuerpo

> **Para el estudiante:** Este notebook fue diseñado para que puedas aprender
> leyendo el código. Cada línea tiene un comentario que explica QUÉ hace,
> POR QUÉ se hace así, y qué pasaría si la cambiaras. No necesitas un maestro
> al lado — el código es tu maestro.

---

## 🎯 ¿Qué aprenderás?

| # | Habilidad |
|---|---|
| 1 | Qué son los **landmarks** y cómo se organizan en rostro, mano y cuerpo |
| 2 | Usar **MediaPipe** para detección en imágenes estáticas |
| 3 | Extraer coordenadas **x, y, z** de cada punto clave |
| 4 | Calcular **ángulos articulares** para análisis biomecánico |
| 5 | Detectar **gestos con la mano** usando lógica geométrica |
| 6 | Detectar **expresiones faciales** (somnolencia, boca abierta) |
| 7 | Detectar **caídas y posturas incorrectas** |
| 8 | Usar **YOLOv8-pose** como tecnología alternativa |
| 9 | Aplicación en **tiempo real** con webcam |

---

## 🗺️ Estructura del Notebook

```
MÓDULO 0  → Instalación, conceptos base y funciones utilitarias
MÓDULO 1  → Face Mesh (468 puntos del rostro)
MÓDULO 2  → Hand Landmarks (21 puntos por mano)
MÓDULO 3  → Body Pose (33 puntos del cuerpo + ángulos)
MÓDULO 4  → Holistic (rostro + manos + cuerpo simultáneo)
MÓDULO 5  → Detección de gestos con la mano
MÓDULO 6  → Detección de caídas y posturas incorrectas
MÓDULO 7  → Aplicación en tiempo real con webcam
MÓDULO 8  → YOLOv8-pose como alternativa
MÓDULO 9  → Comparativa de tecnologías
MÓDULO 10 → Resumen, ejercicios y próximos pasos
```

---

## 📚 Conceptos Fundamentales — Léelos antes de ejecutar

### ¿Qué es un Landmark?

Un **landmark** (punto clave) es un punto específico del cuerpo humano que la IA
detecta y rastrea. Piensa en él como un 'pin' clavado en una articulación o
rasgo anatómico. Cada landmark tiene **3 coordenadas**:

```
x → posición horizontal   (0.0 = izquierda de la imagen,  1.0 = derecha)
y → posición vertical     (0.0 = arriba de la imagen,     1.0 = abajo)
z → profundidad           (valor negativo = más cerca de la cámara)
```

> 💡 **Importante:** Los valores de x e y son **normalizados** (entre 0.0 y 1.0).
> Esto significa que funcionan igual sin importar si la imagen es de 640x480
> o de 1920x1080. Para convertirlos a píxeles reales:
>   `pixel_x = landmark.x * ancho_imagen`
>   `pixel_y = landmark.y * alto_imagen`

### ¿Qué es la Estimación de Pose?

Es el proceso de localizar landmarks en imágenes usando **redes neuronales**
entrenadas con millones de imágenes etiquetadas a mano por humanos.
La red neuronal aprendió a 'ver' articulaciones igual que tú aprendes a
reconocer objetos.

### Pipeline (flujo de trabajo) de cualquier sistema de pose:

```
IMAGEN o VIDEO
      ↓
DETECCIÓN: ¿hay una persona/mano/cara en la imagen?
      ↓
LOCALIZACIÓN: ¿dónde están exactamente los landmarks?
      ↓
POST-PROCESAMIENTO: calcular ángulos, detectar gestos,
                    lanzar alertas, visualizar resultados
```

---

> ✅ Ejecuta las celdas en orden con `Shift + Enter`
> 🔁 Si reinicias el kernel (Runtime → Restart), empieza desde el Módulo 0


---
# 📦 MÓDULO 0 — Instalación y Configuración del Entorno

## ¿Por qué instalamos versiones específicas?

En Python, las librerías evolucionan constantemente y no siempre son
compatibles entre sí. Si instalamos versiones incorrectas, el código
puede fallar con errores crípticos. Por eso **fijamos versiones exactas**.

| Librería | ¿Para qué sirve en este notebook? |
|---|---|
| `mediapipe` | Motor principal de detección de landmarks |
| `opencv-python` | Captura de cámara, lectura de imágenes y dibujo |
| `numpy` | Operaciones matemáticas con vectores y matrices |
| `matplotlib` | Graficar resultados y mostrar imágenes |
| `Pillow` | Abrir y convertir imágenes |
| `pandas` | Organizar coordenadas en tablas legibles |
| `ultralytics` | YOLOv8 (módulo 8 de este notebook) |
| `ipywidgets` | Botones y controles interactivos en Jupyter |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — CELDA 1: INSTALACIÓN DE LIBRERÍAS                  ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es %pip? ────────────────────────────────────────────────
# %pip es un 'magic command' de Jupyter que instala paquetes Python.
# Es como escribir 'pip install ...' en la terminal, pero desde dentro
# del notebook. El prefijo % indica que es un comando especial de Jupyter.

# ─── El flag -q ───────────────────────────────────────────────────
# -q significa 'quiet' (silencioso). Sin él, la instalación imprimiría
# cientos de líneas de información. Con -q, solo vemos lo esencial.

# ─── ¿Por qué mediapipe==0.10.21 específicamente? ─────────────────
# mediapipe 0.10.21 es la última versión que incluye mp.solutions.*
# (la API que usamos aquí). Versiones más nuevas cambiaron la estructura.
# En producción real siempre fija versiones para garantizar reproducibilidad.

# ─── ¿Por qué numpy>=1.24,<2? ─────────────────────────────────────
# mediapipe 0.10.21 no es compatible con numpy 2.x (cambió la API interna).
# El operador >= significa 'mayor o igual que', < significa 'menor que'.
# Juntos forman un rango: 'instala cualquier numpy entre 1.24 y 1.x (sin 2)'.

# ─── ¿Por qué protobuf>=3.20,<4? ──────────────────────────────────
# protobuf es una librería de serialización que MediaPipe usa internamente.
# La versión 4+ cambió la API interna de 'MessageFactory' y rompe mediapipe.

%pip install -q "mediapipe==0.10.21" "numpy>=1.24,<2" "protobuf>=3.20,<4"

# Instalamos las demás librerías sin restricciones de versión
# porque son más estables y no tienen conflictos entre sí.
%pip install -q opencv-python matplotlib Pillow pandas ipywidgets

# ultralytics contiene YOLOv8. Se instala por separado porque es
# un paquete grande (~200MB) y no siempre es necesario.
%pip install -q ultralytics

print("✅ Instalación completada.")
print("ℹ️  Si alguna librería falla al importar más adelante, reinicia el kernel.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — CELDA 2: IMPORTACIONES GLOBALES                    ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es 'importar'? ──────────────────────────────────────────
# Python instala las librerías en el disco (con pip install), pero para
# usarlas en el código hay que 'importarlas' a la memoria del programa.
# 'import X' carga toda la librería.
# 'from X import Y' carga solo la parte Y (más eficiente).

# ─── Librerías estándar de Python (ya vienen instaladas) ──────────
import os           # interactúa con el sistema de archivos (verificar si existe un archivo, etc.)
import io           # manejo de flujos de bytes (necesario para leer imágenes desde memoria)
import time         # funciones de tiempo: time.time(), time.sleep()
import math         # funciones matemáticas: math.degrees(), math.acos(), math.sqrt()
import threading    # permite ejecutar varias tareas simultáneamente (necesario para la webcam)
import warnings     # sistema de advertencias de Python
import urllib.request  # descarga archivos desde internet

# Suprimimos advertencias de deprecación para no contaminar la salida del notebook.
# En proyectos serios deberías investigar y resolver cada warning, no ignorarlos.
warnings.filterwarnings('ignore')

# ─── Librerías de visión artificial ───────────────────────────────
import cv2          # OpenCV: captura de cámara, lectura/escritura de imágenes, dibujo
                    # cv2 viene del nombre original 'Computer Vision 2'

import numpy as np  # NumPy: arrays multidimensionales (imágenes son arrays de píxeles)
                    # 'as np' es un alias — en lugar de escribir numpy.array() escribimos np.array()

import mediapipe as mp  # MediaPipe: el motor de detección de landmarks de Google
                        # 'as mp' → mp.solutions.pose, mp.Image, etc.

from PIL import Image  # Pillow: abre archivos .jpg, .png, .webp y los convierte a arrays
                       # PIL = Python Imaging Library (nombre original, Pillow es el fork moderno)

# ─── Librerías de visualización ───────────────────────────────────
import matplotlib.pyplot as plt         # 'plt' es el alias estándar para crear gráficas
import matplotlib.patches as mpatches   # para crear formas y leyendas personalizadas
from mpl_toolkits.mplot3d import Axes3D # extensión 3D para matplotlib (gráficas en 3 ejes)

# ─── Librerías de datos e interfaz ────────────────────────────────
import pandas as pd                      # pandas: tablas de datos (DataFrames)
                                         # 'as pd' es el alias estándar

import ipywidgets as widgets             # widgets interactivos en Jupyter: botones, sliders, etc.

from IPython.display import display, clear_output, HTML
# display(): muestra cualquier objeto en el notebook (DataFrames, imágenes, HTML, etc.)
# clear_output(): borra la salida actual de una celda (útil para actualizar en tiempo real)
# HTML(): renderiza código HTML en el notebook

# ─── Nueva API de MediaPipe: mp.tasks ─────────────────────────────
#
# MediaPipe tiene DOS APIs. Aquí usamos la NUEVA (mp.tasks):
#
#   API LEGACY (antigua):           API NUEVA (moderna):
#   mp.solutions.pose.Pose()        PoseLandmarkerOptions()
#   detector.process(imagen_rgb)    detector.detect(mp.Image(...))
#   result.pose_landmarks.landmark  result.pose_landmarks[0]
#
# La nueva API es más robusta, más rápida y mejor documentada.
# Los modelos son archivos .task que se descargan por separado.

from mediapipe.tasks.python import vision  # módulo de visión de la nueva API
from mediapipe.tasks.python.core import base_options as base_opts_module
# base_options define DÓNDE está el modelo: base_options.model_asset_path = 'archivo.task'

from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
# mp_drawing.draw_landmarks() dibuja los puntos y conexiones sobre la imagen

from mediapipe.tasks.python.vision import drawing_styles as mp_drawing_styles
# mp_drawing_styles contiene estilos visuales predefinidos (colores, grosores)
# Para rostro: get_default_face_mesh_contours_style()
# Para manos:  get_default_hand_landmarks_style()
# Para cuerpo: get_default_pose_landmarks_style()

from mediapipe.tasks.python.vision import (
    # ── Detectores ──────────────────────────────────────────────────
    FaceLandmarker,          # detecta y localiza landmarks del rostro
    FaceLandmarkerOptions,   # configuración del detector de rostro
    FaceLandmarksConnections,# defines qué landmarks conectar visualmente

    HandLandmarker,          # detecta y localiza landmarks de la mano
    HandLandmarkerOptions,   # configuración del detector de mano
    HandLandmarksConnections,# conexiones del esqueleto de la mano

    PoseLandmarker,          # detecta y localiza landmarks del cuerpo
    PoseLandmarkerOptions,   # configuración del detector de cuerpo
    PoseLandmarksConnections,# conexiones del esqueleto del cuerpo

    GestureRecognizer,       # detecta manos Y clasifica gestos con ML
    GestureRecognizerOptions,# configuración del clasificador de gestos

    RunningMode,             # modo de operación del detector:
                             #   IMAGE      → imagen estática (un solo frame)
                             #   VIDEO      → video con timestamps (tracking entre frames)
                             #   LIVE_STREAM → cámara en tiempo real (callback asíncrono)
)

# Renombramos BaseOptions para escribir menos en el resto del código
BaseOptions = base_opts_module.BaseOptions

print("✅ Todas las importaciones exitosas.")
print(f"   OpenCV versión     : {cv2.__version__}")
print(f"   NumPy versión      : {np.__version__}")
print(f"   MediaPipe versión  : {mp.__version__}")
print(f"   Pandas versión     : {pd.__version__}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — CELDA 3: DESCARGA DE MODELOS .task                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es un archivo .task? ────────────────────────────────────
# La nueva API de MediaPipe NO incluye los modelos dentro del paquete pip.
# Cada detector necesita su propio archivo .task que contiene:
#   1. La arquitectura de la red neuronal
#   2. Los pesos entrenados (los 'conocimientos' del modelo)
#   3. Metadatos (nombres de clases, configuraciones, etc.)
# Son esencialmente modelos TFLite (TensorFlow Lite) empacados.

# ─── ¿Por qué hay 3 variantes de pose? ────────────────────────────
# lite  → modelo pequeño y rápido, menos preciso  (~6MB, ~30fps en CPU)
# full  → modelo balanceado — recomendado para uso general (~9MB, ~15fps)
# heavy → modelo más grande y preciso, más lento (~30MB, ~5fps en CPU)
# Elige según tu caso: si necesitas velocidad en tiempo real → lite
#                      si necesitas precisión para análisis → heavy

# ─── Diccionario de modelos ───────────────────────────────────────
# Un diccionario en Python es una colección de pares 'clave: valor'.
# Sintaxis: {'clave1': valor1, 'clave2': valor2, ...}
# Aquí cada clave es un nombre amigable y el valor es una tupla (nombre_archivo, url).

MODELOS = {
    # Sintaxis de cada entrada:
    # 'nombre_clave': ('nombre_archivo_local.task', 'https://url_del_modelo')

    'face': ('face_landmarker.task',
             'https://storage.googleapis.com/mediapipe-models/'
             'face_landmarker/face_landmarker/float16/1/face_landmarker.task'),
    # 'face' → detecta 468 landmarks del rostro
    # float16 en la URL indica precisión de 16 bits (más rápido que 32, casi igual de preciso)

    'hand': ('hand_landmarker.task',
             'https://storage.googleapis.com/mediapipe-models/'
             'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'),
    # 'hand' → detecta 21 landmarks por mano (hasta 2 manos)

    'pose_lite': ('pose_landmarker_lite.task',
                  'https://storage.googleapis.com/mediapipe-models/'
                  'pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'),
    # pose_lite → 33 landmarks del cuerpo, modelo pequeño (~6MB)

    'pose_full': ('pose_landmarker_full.task',
                  'https://storage.googleapis.com/mediapipe-models/'
                  'pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task'),
    # pose_full → 33 landmarks del cuerpo, modelo balanceado (~9MB)

    'pose_heavy': ('pose_landmarker_heavy.task',
                   'https://storage.googleapis.com/mediapipe-models/'
                   'pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task'),
    # pose_heavy → 33 landmarks del cuerpo, modelo pesado (~30MB)

    'gesture': ('gesture_recognizer.task',
                'https://storage.googleapis.com/mediapipe-models/'
                'gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'),
    # gesture → detecta manos Y reconoce gestos (puño, paz, pulgar, etc.)
}

print("⬇️  Verificando y descargando modelos MediaPipe...")
print("   (Los archivos se guardan localmente para no re-descargarlos)")
print()

# ─── Bucle de descarga ────────────────────────────────────────────
# .items() en un diccionario devuelve pares (clave, valor).
# En cada iteración, 'nombre' = la clave y '(archivo, url)' = la tupla del valor.
for nombre, (archivo, url) in MODELOS.items():

    # ─── os.path.exists(archivo) ──────────────────────────────────
    # Verifica si el archivo ya existe en el directorio actual.
    # os.path.getsize(archivo) devuelve el tamaño en bytes.
    # Si el archivo existe Y tiene más de 1000 bytes → ya fue descargado correctamente.
    # (Un archivo de 0 bytes indicaría una descarga fallida anterior.)
    if os.path.exists(archivo) and os.path.getsize(archivo) > 1000:
        print(f"   ✅ {archivo} ya existe ({os.path.getsize(archivo)//1024} KB)")
        # // es división entera (sin decimales), convierte bytes a kilobytes
    else:
        # end=' ' evita el salto de línea al final, flush=True fuerza que se imprima ahora
        print(f"   ⬇️  Descargando {archivo}...", end=' ', flush=True)
        try:
            # urllib.request.urlretrieve(url, destino) descarga url y lo guarda en destino
            urllib.request.urlretrieve(url, archivo)
            kb = os.path.getsize(archivo) // 1024  # tamaño en KB
            print(f"✅ {kb} KB")
        except Exception as e:
            # Exception captura CUALQUIER tipo de error en Python.
            # 'as e' guarda el error en la variable 'e' para poder imprimirlo.
            print(f"❌ Error: {e}")

# ─── Variables de ruta (shortcuts) ────────────────────────────────
# En lugar de escribir MODELOS['face'][0] cada vez que necesitamos la ruta,
# creamos variables con nombres cortos y descriptivos.
MODEL_FACE       = MODELOS['face'][0]       # → 'face_landmarker.task'
MODEL_HAND       = MODELOS['hand'][0]       # → 'hand_landmarker.task'
MODEL_POSE_LITE  = MODELOS['pose_lite'][0]  # → 'pose_landmarker_lite.task'
MODEL_POSE_FULL  = MODELOS['pose_full'][0]  # → 'pose_landmarker_full.task'
MODEL_POSE_HEAVY = MODELOS['pose_heavy'][0] # → 'pose_landmarker_heavy.task'
MODEL_GESTURE    = MODELOS['gesture'][0]    # → 'gesture_recognizer.task'

print()
print("✅ Modelos listos. Variables de ruta creadas:")
print("   MODEL_FACE, MODEL_HAND, MODEL_POSE_LITE, MODEL_POSE_FULL,")
print("   MODEL_POSE_HEAVY, MODEL_GESTURE")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — CELDA 4: FUNCIONES UTILITARIAS                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es una función utilitaria? ─────────────────────────────
# Son pequeñas funciones de ayuda que resuelven problemas comunes que
# se repiten en muchas partes del código. En lugar de escribir el mismo
# código 20 veces, lo escribimos una vez aquí y lo llamamos cuando sea necesario.
# Principio DRY: Don't Repeat Yourself (No te repitas).

# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 1: bgr_a_rgb()
# ════════════════════════════════════════════════════════════════════

def bgr_a_rgb(imagen_bgr):
    """
    Convierte imagen del formato BGR (OpenCV) al formato RGB (MediaPipe/Matplotlib).

    ¿Por qué existe esta incompatibilidad?
    OpenCV fue creado hace 20+ años cuando el hardware almacenaba los canales
    de color en orden B-G-R por razones históricas de eficiencia.
    Todas las demás librerías modernas usan R-G-B (el orden natural).

    Si no conviertes antes de mostrar:
      → Los colores aparecerán invertidos (lo rojo se ve azul y viceversa).

    Parámetro:
        imagen_bgr → array NumPy de forma (alto, ancho, 3) en orden BGR

    Retorna:
        array NumPy de forma (alto, ancho, 3) en orden RGB
    """
    # cv2.cvtColor(imagen, codigo_conversion) convierte entre espacios de color.
    # cv2.COLOR_BGR2RGB es una constante que indica 'de BGR a RGB'.
    # Hay muchas otras: COLOR_BGR2GRAY, COLOR_BGR2HSV, COLOR_RGB2LAB, etc.
    return cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 2: rgb_a_mp_image()
# ════════════════════════════════════════════════════════════════════

def rgb_a_mp_image(imagen_rgb):
    """
    Convierte un array NumPy RGB al formato mp.Image que requiere la nueva API.

    ¿Por qué necesitamos esto?
    La API LEGACY (antigua) aceptaba NumPy directamente:
        detector.process(imagen_rgb)  ← array NumPy directo

    La API NUEVA requiere un 'wrapper' especial:
        detector.detect(mp.Image(...)) ← objeto mp.Image

    mp.Image es como una 'caja' que empaqueta el array NumPy junto con
    metadatos sobre el formato de color.

    Parámetro:
        imagen_rgb → array NumPy RGB de forma (alto, ancho, 3)

    Retorna:
        objeto mp.Image listo para pasar a los detectores
    """
    # mp.Image necesita dos argumentos:
    #   image_format=mp.ImageFormat.SRGB → indica que los canales están en orden R-G-B
    #   data=imagen_rgb                   → el array NumPy con los píxeles
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 3: cargar_imagen()
# ════════════════════════════════════════════════════════════════════

def cargar_imagen(ruta_o_url):
    """
    Carga una imagen desde el disco local o desde una URL de internet.
    Siempre devuelve un array NumPy en formato RGB.

    Ejemplos de uso:
        img = cargar_imagen('mi_foto.jpg')          # desde disco
        img = cargar_imagen('https://..../foto.jpg') # desde internet
    """
    # str.startswith('http') verifica si la cadena empieza con 'http'
    # → True para URLs de internet, False para rutas de archivo local
    if ruta_o_url.startswith('http'):
        # Descarga la imagen desde internet como bytes en memoria
        with urllib.request.urlopen(ruta_o_url) as resp:
            datos = resp.read()  # lee todos los bytes de la respuesta HTTP

        # io.BytesIO(datos) crea un 'archivo en memoria' a partir de los bytes
        # Image.open() abre la imagen desde ese archivo en memoria
        # .convert('RGB') garantiza que tenga 3 canales RGB (incluso si era RGBA o escala de grises)
        return np.array(Image.open(io.BytesIO(datos)).convert('RGB'))
        # np.array() convierte la imagen PIL a array NumPy
    else:
        # cv2.imread() lee una imagen del disco. Devuelve array BGR.
        img_bgr = cv2.imread(ruta_o_url)

        # Si la imagen no se encontró, cv2.imread() devuelve None (no lanza error).
        # Verificamos esto explícitamente para dar un mensaje útil al usuario.
        if img_bgr is None:
            raise FileNotFoundError(f"No se encontró el archivo: {ruta_o_url}")
            # raise lanza una excepción (error) para detener la ejecución con un mensaje claro.

        return bgr_a_rgb(img_bgr)  # convertimos BGR → RGB antes de devolver


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 4: mostrar_imagen()
# ════════════════════════════════════════════════════════════════════

def mostrar_imagen(img_rgb, titulo='Imagen', figsize=(10, 6)):
    """
    Muestra una imagen RGB en el notebook usando Matplotlib.

    ¿Por qué usamos Matplotlib y no cv2.imshow()?
    cv2.imshow() abre una ventana externa del sistema operativo,
    lo que no funciona en Jupyter/Google Colab.
    Matplotlib sí muestra imágenes inline (dentro del notebook).

    Parámetros:
        img_rgb  → array NumPy RGB
        titulo   → texto que aparece sobre la imagen
        figsize  → tamaño de la figura en pulgadas (ancho, alto)
    """
    # plt.figure() crea un nuevo lienzo de matplotlib con el tamaño especificado.
    plt.figure(figsize=figsize)

    # plt.imshow() renderiza el array NumPy como imagen.
    # Matplotlib espera RGB — por eso siempre convertimos desde BGR antes de llegar aquí.
    plt.imshow(img_rgb)

    # plt.title() agrega texto en la parte superior de la imagen.
    # fontsize=13 define el tamaño del texto.
    # fontweight='bold' lo pone en negrita.
    plt.title(titulo, fontsize=13, fontweight='bold')

    # plt.axis('off') oculta los ejes X e Y (no necesitamos coordenadas de píxeles en los bordes).
    plt.axis('off')

    # plt.tight_layout() ajusta automáticamente el espaciado para que nada quede cortado.
    plt.tight_layout()

    # plt.show() renderiza y muestra la imagen en el notebook.
    plt.show()


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 5: mostrar_comparacion()
# ════════════════════════════════════════════════════════════════════

def mostrar_comparacion(img1, img2, titulo1='Original', titulo2='Procesada', figsize=(16, 7)):
    """
    Muestra dos imágenes lado a lado para facilitar la comparación.
    Útil para ver la imagen original vs la imagen con landmarks dibujados.
    """
    # plt.subplots(filas, columnas) crea una cuadrícula de gráficas.
    # 1 fila, 2 columnas → dos imágenes lado a lado.
    # Retorna: fig (el lienzo completo) y axes (lista de los 2 ejes individuales).
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # axes[0] → panel izquierdo (imagen 1)
    # axes[1] → panel derecho  (imagen 2)
    axes[0].imshow(img1)
    axes[0].set_title(titulo1, fontsize=12, fontweight='bold')
    axes[0].axis('off')  # oculta ejes del panel izquierdo

    axes[1].imshow(img2)
    axes[1].set_title(titulo2, fontsize=12, fontweight='bold')
    axes[1].axis('off')  # oculta ejes del panel derecho

    plt.tight_layout()
    plt.show()


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 6: calcular_angulo() — MUY IMPORTANTE para biomecánica
# ════════════════════════════════════════════════════════════════════

def calcular_angulo(punto_a, punto_b, punto_c):
    """
    Calcula el ángulo en grados formado por 3 puntos en el vértice B.
    Este es el corazón del análisis biomecánico.

    Visualización:
        A •                   Por ejemplo, para el ángulo del codo:
           \                    A = hombro (11)
            \  ángulo           B = codo   (13)  ← vértice
             • B                C = muñeca (15)
            /
           /
        C •

    Matemática (producto punto / dot product):
        cos(θ) = (BA · BC) / (|BA| × |BC|)
        θ = acos(cos(θ))

    Parámetros:
        punto_a → [x, y] del punto A (puede incluir z como tercer elemento, se ignora)
        punto_b → [x, y] del vértice B (donde se mide el ángulo)
        punto_c → [x, y] del punto C

    Retorna:
        ángulo en grados (float)
        0° = puntos sobre la misma recta
        90° = ángulo recto
        180° = completamente extendido (recta)
    """
    # punto[:2] toma solo los primeros 2 elementos del array.
    # Esto permite pasar puntos con z (3D) o sin z (2D) — ignoramos z.
    a = np.array(punto_a[:2])  # convierte la lista [x,y] a array NumPy
    b = np.array(punto_b[:2])  # el vértice donde se mide el ángulo
    c = np.array(punto_c[:2])

    # Calculamos los vectores que apuntan DESDE B hacia A y hacia C.
    # Vector = diferencia de posiciones = destino - origen
    ba = a - b  # vector del vértice B al punto A
    bc = c - b  # vector del vértice B al punto C

    # np.linalg.norm() calcula la magnitud (longitud) del vector.
    # Es la fórmula de la distancia euclidiana: sqrt(x² + y²)
    mag_ba = np.linalg.norm(ba)  # longitud del vector BA
    mag_bc = np.linalg.norm(bc)  # longitud del vector BC

    # Si algún vector tiene longitud 0, significa que dos puntos coinciden.
    # La división entre 0 causaría un error, así que devolvemos 0.
    if mag_ba == 0 or mag_bc == 0:
        return 0.0

    # np.dot(ba, bc) → producto punto: ba.x*bc.x + ba.y*bc.y
    # Dividimos por las magnitudes para obtener el coseno del ángulo.
    # np.clip(..., -1.0, 1.0) limita el valor al rango [-1, 1].
    # Esto previene errores numéricos: a veces el cálculo da 1.0000001 por precisión flotante,
    # y acos(1.0000001) causaría un error. clip() lo corrige.
    coseno = np.clip(np.dot(ba, bc) / (mag_ba * mag_bc), -1.0, 1.0)

    # math.acos() = arcocoseno → devuelve el ángulo en RADIANES.
    # math.degrees() convierte RADIANES a GRADOS.
    # round(..., 1) redondea a 1 decimal.
    return round(math.degrees(math.acos(coseno)), 1)


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 7: landmark_a_pixeles()
# ════════════════════════════════════════════════════════════════════

def landmark_a_pixeles(landmark, alto, ancho):
    """
    Convierte las coordenadas normalizadas (0.0 a 1.0) de un landmark
    a coordenadas absolutas en píxeles.

    ¿Por qué es necesario?
    MediaPipe usa coordenadas normalizadas para independencia de resolución.
    Pero cv2.circle(), cv2.putText() y similares necesitan píxeles absolutos.

    Ejemplo:
        landmark.x = 0.5, landmark.y = 0.3
        imagen de 640x480 píxeles
        → pixel_x = int(0.5 * 640) = 320
        → pixel_y = int(0.3 * 480) = 144

    Parámetros:
        landmark → objeto NormalizedLandmark con atributos .x y .y (ambos entre 0.0 y 1.0)
        alto     → altura de la imagen en píxeles
        ancho    → ancho de la imagen en píxeles

    Retorna:
        tupla (x_pixeles, y_pixeles) de enteros
    """
    # int() convierte float a entero truncando los decimales (ej: 319.7 → 319)
    # Esto es necesario porque cv2 requiere coordenadas enteras para dibujar.
    return (int(landmark.x * ancho), int(landmark.y * alto))


print("✅ Funciones utilitarias listas:")
print("   bgr_a_rgb()        → convierte BGR (OpenCV) a RGB")
print("   rgb_a_mp_image()   → convierte NumPy RGB a mp.Image")
print("   cargar_imagen()    → carga imagen de disco o URL")
print("   mostrar_imagen()   → muestra imagen en el notebook")
print("   mostrar_comparacion() → muestra dos imágenes lado a lado")
print("   calcular_angulo()  → ángulo en grados dado 3 puntos (A-B-C)")
print("   landmark_a_pixeles() → convierte coord. normalizadas a píxeles")


---
# 😊 MÓDULO 1 — Face Mesh: 468 Landmarks del Rostro

## ¿Qué es Face Mesh?

MediaPipe Face Mesh detecta **468 puntos tridimensionales** en el rostro humano.
Imagina que pones 468 'alfileres' en puntos específicos de la cara: cejas, ojos,
nariz, boca, contorno... eso es exactamente lo que hace el modelo.

```
ZONAS CON SUS ÍNDICES CLAVE:

  👁️  Ojo izquierdo  → landmarks 33, 133, 159, 145   (borde del párpado)
  👁️  Ojo derecho    → landmarks 362, 263, 386, 374   (borde del párpado)
  👃  Nariz          → landmark 1 (punta), 6 (puente)
  👄  Boca           → landmarks 13 (labio sup), 14 (labio inf), 61, 291 (comisuras)
  🧔  Cara/Contorno  → landmarks 151 (frente), 152 (mentón), 234, 454 (mejillas)
```

> 💡 **Nota:** Los índices 0-467 son FIJOS. El índice 1 siempre es la punta de la nariz,
> el 152 siempre es el mentón, etc. Esto es por diseño para que el código sea predecible.

## Aplicaciones reales
- 😴 Detección de somnolencia (EAR — Eye Aspect Ratio)
- 🗣️ Detección de boca abierta (MAR — Mouth Aspect Ratio)
- 🚗 Sistemas de alerta para conductores cansados
- 🎭 Filtros de realidad aumentada (Snapchat, Instagram)
- 😶 Análisis de expresiones faciales


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.A — MAPA DE LANDMARKS DEL ROSTRO                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Por qué hacer un diccionario en lugar de usar números directamente? ──
# Comparación:
#   MALO (números sin contexto):
#       lm = face_lms[159]   ← ¿qué es el punto 159?
#
#   BUENO (nombre descriptivo):
#       lm = face_lms[FACE_LANDMARKS['ojo_izq_arriba']]  ← claro y autodocumentado
#
# Un diccionario {'nombre': índice} hace el código legible sin perder eficiencia.

FACE_LANDMARKS = {
    # ── OJOS ──────────────────────────────────────────────────────
    # Los 4 puntos extremos de cada ojo forman una 'X' alrededor del ojo.
    # Con ellos calculamos el EAR (Eye Aspect Ratio):
    #   EAR = (distancia_vertical_1 + distancia_vertical_2) / (2 * distancia_horizontal)
    #   EAR cercano a 0 → ojo cerrado | EAR > 0.20 → ojo abierto

    'ojo_izq_arriba':     159,  # párpado superior del ojo izquierdo
    'ojo_izq_abajo':      145,  # párpado inferior del ojo izquierdo
    'ojo_izq_izquierda':   33,  # comisura externa (lado de la oreja)
    'ojo_izq_derecha':    133,  # comisura interna (lado de la nariz)

    'ojo_der_arriba':     386,  # párpado superior del ojo derecho
    'ojo_der_abajo':      374,  # párpado inferior del ojo derecho
    'ojo_der_izquierda':  362,  # comisura interna del ojo derecho
    'ojo_der_derecha':    263,  # comisura externa del ojo derecho

    # ── BOCA ──────────────────────────────────────────────────────
    # Con los 4 puntos de la boca calculamos el MAR (Mouth Aspect Ratio):
    #   MAR = distancia_vertical / distancia_horizontal
    #   MAR > 0.50 → boca abierta (bostezo, habla, sorpresa)

    'labio_arriba':       13,   # labio superior, punto central
    'labio_abajo':        14,   # labio inferior, punto central
    'boca_izquierda':     61,   # comisura izquierda de la boca
    'boca_derecha':       291,  # comisura derecha de la boca

    # ── NARIZ ─────────────────────────────────────────────────────
    'punta_nariz':        1,    # punta de la nariz (landmark muy estable)
    'nariz_puente':       6,    # entre los ojos, en el hueso nasal

    # ── CARA / GEOMETRÍA FACIAL ───────────────────────────────────
    'frente_centro':      151,  # punto central de la frente
    'menton':             152,  # punta del mentón
    'mejilla_izquierda':  234,  # mejilla izquierda (punto extremo)
    'mejilla_derecha':    454,  # mejilla derecha (punto extremo)

    # ── CEJAS ─────────────────────────────────────────────────────
    'ceja_izq_centro':    70,   # punto central de la ceja izquierda
    'ceja_der_centro':    300,  # punto central de la ceja derecha
}

print(f"✅ FACE_LANDMARKS definido con {len(FACE_LANDMARKS)} puntos clave.")
print(f"   (De los 468 landmarks totales del rostro)")
print()

# ─── Mostrar la tabla organizada por zona ─────────────────────────
# Usamos comprensiones de listas para filtrar el diccionario por zona:
# [{'Zona': ..., 'Nombre': k, 'Índice': v} for k, v in FACE_LANDMARKS.items() if condición]
# Esto itera sobre el diccionario y crea una lista de filas para el DataFrame.

df_face = pd.DataFrame([
    {'Zona': 'Ojo',   'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'ojo' in k
] + [
    {'Zona': 'Boca',  'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'labio' in k or 'boca' in k
] + [
    {'Zona': 'Nariz', 'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'nariz' in k
] + [
    {'Zona': 'Cara',  'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items()
    if k not in [k2 for k2 in FACE_LANDMARKS if 'ojo' in k2 or 'labio' in k2 or 'boca' in k2 or 'nariz' in k2]
])

print("📊 Tabla de landmarks clave — referencia anatómica:")
display(df_face)
print()
print("💡 CONSEJO: Guarda esta tabla como referencia al programar.")
print("   Así no necesitas memorizar los números — solo los nombres.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.B — FUNCIÓN: analizar_rostro_imagen()                 ║
# ╚══════════════════════════════════════════════════════════════════╝

def analizar_rostro_imagen(imagen_rgb, dibujar_malla=True, dibujar_puntos_clave=True):
    """
    Detecta y visualiza los 468 landmarks del rostro en una imagen estática.

    FLUJO DE LA FUNCIÓN:
        1. Configurar el detector con FaceLandmarkerOptions
        2. Convertir la imagen al formato mp.Image
        3. Ejecutar detector.detect()
        4. Verificar si se detectó algún rostro
        5. Dibujar la malla facial sobre la imagen
        6. Resaltar los puntos anatómicos clave
        7. Construir tabla de coordenadas
        8. Devolver imagen anotada + resultados + tabla

    Parámetros:
        imagen_rgb           → array NumPy RGB (alto × ancho × 3)
        dibujar_malla        → si True, dibuja los 468 puntos en malla
        dibujar_puntos_clave → si True, resalta los puntos de FACE_LANDMARKS

    Retorna:
        imagen_anotada → imagen con landmarks dibujados (array NumPy RGB)
        resultado      → objeto FaceLandmarkerResult (datos brutos del detector)
        tabla_datos    → DataFrame de pandas con coordenadas de puntos clave
    """
    # Creamos una copia para no modificar la imagen original.
    # Si usáramos imagen_rgb directamente, los dibujos se harían sobre el original.
    # .copy() es fundamental en visión artificial para preservar el original.
    imagen_anotada = imagen_rgb.copy()

    # imagen_rgb.shape devuelve (alto, ancho, canales).
    # [:2] extrae solo los primeros 2 elementos: (alto, ancho).
    # Nota: el orden es (alto, ancho) en NumPy, pero (ancho, alto) en matemáticas.
    # ¡Esto causa muchos bugs! Siempre sé explícito con los nombres.
    alto, ancho = imagen_rgb.shape[:2]

    # ── PASO 1: Configurar el detector ────────────────────────────
    # FaceLandmarkerOptions es el objeto de configuración del detector.
    # Piensa en él como un formulario que llenas antes de crear el detector.
    opciones = FaceLandmarkerOptions(

        # BaseOptions le dice al detector DÓNDE está el modelo.
        # model_asset_path = ruta al archivo .task descargado en el Módulo 0.
        base_options=BaseOptions(model_asset_path=MODEL_FACE),

        # running_mode define cómo se usará el detector:
        #   IMAGE      → para imágenes individuales (este módulo)
        #   VIDEO      → para videos con timestamps
        #   LIVE_STREAM → para cámara en tiempo real con callbacks
        running_mode=RunningMode.IMAGE,

        # num_faces: cuántos rostros detectar como máximo.
        # Poner 5 no significa que siempre buscará 5 — simplemente establece el límite superior.
        num_faces=1,

        # min_face_detection_confidence: qué tan seguro debe estar el modelo antes
        # de declarar que detectó un rostro. Rango: 0.0 a 1.0.
        # 0.5 = 50% de confianza mínima (valor por defecto recomendado).
        # Si pones 0.9 → solo detectará rostros muy claros, perderá rostros parciales.
        # Si pones 0.1 → detectará muchas cosas que no son rostros (falsos positivos).
        min_face_detection_confidence=0.5,

        # min_face_presence_confidence: confianza mínima de que un rostro está presente.
        min_face_presence_confidence=0.5,

        # min_tracking_confidence: confianza del tracking entre frames (en modo VIDEO/LIVE_STREAM).
        min_tracking_confidence=0.5,

        # output_face_blendshapes=True generaría 52 coeficientes de expresiones faciales
        # (estándar ARKit de Apple, útil para animación 3D). Lo desactivamos para velocidad.
        output_face_blendshapes=False,
    )

    # ── PASO 2 y 3: Crear detector, convertir imagen y detectar ───
    # 'with' es el gestor de contexto de Python.
    # Garantiza que el detector se cierre correctamente al salir del bloque,
    # liberando memoria. Equivale a: detector = FaceLandmarker.create_from_options(opciones)
    # ... usar detector ...   detector.close()  — pero de forma automática.
    with FaceLandmarker.create_from_options(opciones) as detector:

        # Convertimos la imagen NumPy al formato mp.Image (requerido por la nueva API).
        mp_imagen = rgb_a_mp_image(imagen_rgb)

        # detector.detect() ejecuta la inferencia (el modelo de red neuronal).
        # Devuelve un objeto FaceLandmarkerResult con todos los resultados.
        resultado = detector.detect(mp_imagen)

        # ── PASO 4: Verificar si se detectó algún rostro ──────────
        # resultado.face_landmarks es una lista de listas:
        #   - La lista externa tiene un elemento por CADA ROSTRO detectado.
        #   - Cada elemento interno es una lista de 468 NormalizedLandmark.
        # Si la lista está vacía (no se detectó ningún rostro), su valor es falsy.
        if not resultado.face_landmarks:
            print("⚠️  No se detectó ningún rostro en la imagen.")
            print("   Sugerencias: usa una imagen más clara, más iluminada,")
            print("   o con el rostro más frontal (de frente a la cámara).")
            # Retornamos valores vacíos para que el código que llama a esta
            # función pueda verificar si hubo resultados.
            return imagen_anotada, None, pd.DataFrame()

        print(f"✅ {len(resultado.face_landmarks)} rostro(s) detectado(s).")

        # ── PASO 5: Procesar cada rostro detectado ────────────────
        # enumerate() devuelve (índice, valor) en cada iteración.
        # idx_rostro: 0, 1, 2... (número del rostro)
        # face_lms: lista plana de 468 NormalizedLandmark del rostro actual
        for idx_rostro, face_lms in enumerate(resultado.face_landmarks):

            # face_lms es una LISTA PLANA de 468 objetos.
            # face_lms[j].x → coordenada x del landmark j (0.0 a 1.0)
            # face_lms[j].y → coordenada y del landmark j (0.0 a 1.0)
            # face_lms[j].z → profundidad del landmark j

            if dibujar_malla:
                # ── Dibuja la MALLA COMPLETA (tessellation) ─────────
                # FACE_LANDMARKS_TESSELATION son los 468 puntos conectados
                # en triángulos — da el efecto de 'malla 3D' sobre el rostro.
                mp_drawing.draw_landmarks(
                    image=imagen_anotada,      # imagen sobre la que se dibuja (se modifica)
                    landmark_list=face_lms,    # la lista de 468 landmarks
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                    # ↑ Opciones de connections:
                    #   FACE_LANDMARKS_TESSELATION → malla densa completa
                    #   FACE_LANDMARKS_CONTOURS    → solo el contorno exterior
                    #   FACE_LANDMARKS_FACE_OVAL   → solo el óvalo facial
                    landmark_drawing_spec=mp_drawing.DrawingSpec(
                        color=(0, 255, 0),  # color en BGR: verde puro
                        thickness=1,       # grosor del punto
                        circle_radius=1    # tamaño del círculo en cada punto
                    ),
                    connection_drawing_spec=mp_drawing.DrawingSpec(
                        color=(0, 200, 100),  # verde más oscuro para las líneas
                        thickness=1
                    ),
                )

                # ── Dibuja el CONTORNO con estilo predefinido ────────
                # get_default_face_mesh_contours_style() devuelve un estilo
                # visual prediseñado que resalta las cejas, ojos y labios.
                mp_drawing.draw_landmarks(
                    image=imagen_anotada,
                    landmark_list=face_lms,
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                    landmark_drawing_spec=None,  # None = no dibuja los puntos, solo las conexiones
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style(),
                )

            if dibujar_puntos_clave:
                # ── Resalta los puntos anatómicos de FACE_LANDMARKS ──
                # Iteramos sobre el diccionario para resaltar puntos importantes
                # con un color diferente (rojo) para distinguirlos de la malla.
                for nombre, idx in FACE_LANDMARKS.items():

                    # Acceso directo al landmark por índice.
                    # face_lms[idx] devuelve el NormalizedLandmark con .x, .y, .z
                    lm = face_lms[idx]

                    # Convertir coordenadas normalizadas a píxeles para dibujar
                    px, py = landmark_a_pixeles(lm, alto, ancho)

                    # cv2.circle(imagen, centro, radio, color, grosor)
                    # grosor=-1 significa 'relleno' (círculo sólido)
                    cv2.circle(imagen_anotada, (px, py), 4, (255, 50, 50), -1)
                    # (255, 50, 50) en RGB es rojo intenso

                    # cv2.putText(imagen, texto, origen, fuente, escala, color, grosor)
                    # nombre[:8] toma los primeros 8 caracteres del nombre (para que no sea muy largo)
                    cv2.putText(imagen_anotada, nombre[:8], (px+5, py),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.25, (255, 255, 0), 1)
                    # cv2.FONT_HERSHEY_SIMPLEX es una fuente simple sin serifs
                    # (255, 255, 0) en RGB es amarillo

            # ── PASO 7: Construir tabla de coordenadas ────────────
            # Creamos una lista de diccionarios, uno por cada punto clave.
            # pd.DataFrame(lista_de_dicts) convierte la lista en una tabla.
            filas = []
            for nombre, idx in FACE_LANDMARKS.items():
                lm = face_lms[idx]
                px, py = landmark_a_pixeles(lm, alto, ancho)
                filas.append({
                    'Nombre':       nombre,
                    'Índice MP':    idx,
                    'x (norm)':     round(lm.x, 4),   # 4 decimales para ver variaciones pequeñas
                    'y (norm)':     round(lm.y, 4),
                    'z (profund)':  round(lm.z, 4),   # negativo = más cercano a la cámara
                    'x (píxeles)':  px,
                    'y (píxeles)':  py,
                })
            tabla_datos = pd.DataFrame(filas)

    return imagen_anotada, resultado, tabla_datos


print("✅ analizar_rostro_imagen() definida y lista para usar.")
print()
print("📌 RECUERDA:")
print("   API legacy: result.multi_face_landmarks[i].landmark[j].x")
print("   API nueva:  result.face_landmarks[i][j].x  ← lista plana")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.C — WIDGET INTERACTIVO: Sube y analiza un rostro     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es un widget? ───────────────────────────────────────────
# Un widget es un elemento de interfaz gráfica (botón, slider, campo de texto)
# que funciona dentro del notebook de Jupyter.
# ipywidgets proporciona estos controles interactivos sin necesitar JavaScript.

# ─── Widget FileUpload ────────────────────────────────────────────
# Crea un botón de 'subir archivo'. El usuario hace clic y selecciona una imagen.
# accept='.jpg,.jpeg,.png' → solo acepta estos formatos (filtro de seguridad)
# multiple=False           → solo permite subir UN archivo a la vez
# description='...'        → texto que aparece en el botón
uploader_rostro = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='📷 Subir foto de rostro'
)

# ─── Widget Button ────────────────────────────────────────────────
# Crea un botón que el usuario puede hacer clic.
# button_style='success' → botón verde (hay: 'success', 'danger', 'warning', 'info', '')
btn_analizar_rostro = widgets.Button(
    description='Analizar Rostro',
    button_style='success'
)

# ─── Widget Output ────────────────────────────────────────────────
# Output() es un contenedor invisible donde se captura la salida (prints, gráficas).
# Todo lo que se ejecute dentro de 'with salida_rostro:' aparece en este contenedor.
# Esto permite actualizar el resultado sin que se mezcle con otras salidas del notebook.
salida_rostro = widgets.Output()

# ─── VBox: alinea widgets verticalmente ──────────────────────────
# VBox (Vertical Box) apila los widgets uno encima del otro.
# HBox (Horizontal Box) los pondría uno al lado del otro.
display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen con un rostro frontal claro:</b>'),
    # HTML('<b>...</b>') permite usar HTML básico para texto en negrita, colores, etc.
    uploader_rostro,
    btn_analizar_rostro,
    salida_rostro
]))


# ─── Función callback (se llama cuando el botón es presionado) ────
# Una función 'callback' no se ejecuta inmediatamente — se registra para
# ejecutarse CUANDO ocurra un evento (en este caso, cuando el usuario presione el botón).
# El parámetro '_' es el objeto del evento (lo ignoramos con _ porque no lo necesitamos).
def on_analizar_rostro(_):
    # 'with salida_rostro:' redirige toda la salida al widget Output.
    with salida_rostro:
        clear_output()  # borra los resultados anteriores para mostrar los nuevos

        # Verificamos que el usuario haya subido algo antes de procesar
        if not uploader_rostro.value:
            print("⚠️  Primero sube una imagen haciendo clic en el botón de arriba.")
            return  # 'return' sin valor sale de la función inmediatamente

        # ─── Extraer los bytes del archivo subido ─────────────────
        # uploader_rostro.value puede ser una lista o un diccionario dependiendo
        # de la versión de ipywidgets. Este código maneja ambos casos.
        valor = uploader_rostro.value

        # isinstance(valor, (list, tuple)) verifica si 'valor' es una lista o tupla.
        # En ipywidgets v8+, value es una lista de objetos.
        # En ipywidgets v7, value es un diccionario {nombre_archivo: {content: bytes}}.
        archivo = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]

        # Los bytes del archivo están en archivo['content'] (dict) o archivo.content (objeto).
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content

        # ─── Convertir bytes → PIL Image → array NumPy RGB ─────────
        # io.BytesIO(contenido) crea un 'archivo virtual' en memoria.
        # Image.open() puede abrir desde ese archivo virtual.
        # .convert('RGB') garantiza 3 canales RGB (por si la imagen es RGBA o grayscale).
        imagen_pil = Image.open(io.BytesIO(contenido)).convert('RGB')

        # np.array() convierte el objeto PIL Image a un array NumPy de forma (alto, ancho, 3).
        imagen_rgb = np.array(imagen_pil)

        print(f"📐 Dimensiones de la imagen: {imagen_rgb.shape[1]}×{imagen_rgb.shape[0]} píxeles")
        # shape[1] = ancho, shape[0] = alto (recuerda: shape es [alto, ancho, canales])
        print()

        # ─── Llamar a la función de análisis ──────────────────────
        imagen_anotada, resultados, tabla = analizar_rostro_imagen(
            imagen_rgb,
            dibujar_malla=True,         # activa la malla completa de 468 puntos
            dibujar_puntos_clave=True   # activa el resaltado de puntos anatómicos
        )

        # ─── Mostrar comparación original vs procesada ─────────────
        mostrar_comparacion(
            imagen_rgb, imagen_anotada,
            'Original', 'Face Mesh — 468 landmarks'
        )

        # ─── Mostrar tabla de coordenadas ──────────────────────────
        if not tabla.empty:  # .empty verifica si el DataFrame no tiene filas
            print("\n📊 Coordenadas de los puntos anatómicos clave:")
            display(tabla)  # display() de IPython renderiza el DataFrame como tabla HTML


# ─── Registrar el callback en el botón ───────────────────────────
# .on_click(función) registra 'on_analizar_rostro' para ejecutarse
# cada vez que el usuario haga clic en 'btn_analizar_rostro'.
btn_analizar_rostro.on_click(on_analizar_rostro)

print("✅ Widget de análisis de rostro listo.")
print("   1. Sube una foto con un rostro bien iluminado y frontal.")
print("   2. Haz clic en 'Analizar Rostro'.")
print("   3. Observa los 468 puntos dibujados sobre el rostro.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.D — EAR y MAR: Somnolencia y Boca Abierta            ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Qué es el EAR (Eye Aspect Ratio)? ──────────────────────────
# El EAR mide qué tan 'abierto' está un ojo comparando su altura con su ancho.

# Fórmula visual:
#
#           p1                    p1=arriba   p4=derecha
#          /  \                   p2=arriba2  p5=abajo2
#   p0 ---    --- p3              p3=izquierda p6=abajo
#          \  /
#           p4     p5
#
# EAR = (||p1-p5|| + ||p2-p4||) / (2 × ||p0-p3||)
# EAR ≈ 0.30 → ojo abierto normal
# EAR < 0.20 → ojo cerrado o casi cerrado (posible somnolencia)

# ─── ¿Qué es el MAR (Mouth Aspect Ratio)? ─────────────────────────
# Similar al EAR pero para la boca:
# MAR = distancia_vertical / distancia_horizontal
# MAR > 0.50 → boca abierta (bostezo, habla, sorpresa)


def calcular_ear(face_lms, puntos_ojo):
    """
    Calcula el Eye Aspect Ratio para un ojo.

    Parámetros:
        face_lms   → lista plana de 468 NormalizedLandmark
        puntos_ojo → lista de 6 índices que definen el ojo:
                     [comisura_izq, arriba_1, arriba_2, comisura_der, abajo_1, abajo_2]

    Retorna:
        EAR como float (0.0 a ~0.40)
    """
    # Extraemos las coordenadas [x, y] de cada uno de los 6 puntos del ojo.
    # Usamos comprensión de lista para hacerlo en una línea.
    # Para cada índice i en la lista puntos_ojo, tomamos face_lms[i].x y .y
    p = [np.array([face_lms[i].x, face_lms[i].y]) for i in puntos_ojo]
    # p[0] = comisura izquierda
    # p[1] = punto superior izquierdo del párpado
    # p[2] = punto superior derecho del párpado
    # p[3] = comisura derecha
    # p[4] = punto inferior izquierdo del párpado
    # p[5] = punto inferior derecho del párpado

    # ─── Distancias verticales ─────────────────────────────────────
    # np.linalg.norm() calcula la distancia euclidiana entre 2 puntos: sqrt((x2-x1)² + (y2-y1)²)
    dist_v1 = np.linalg.norm(p[1] - p[5])  # distancia: arriba_1 ↕ abajo_2
    dist_v2 = np.linalg.norm(p[2] - p[4])  # distancia: arriba_2 ↕ abajo_1

    # ─── Distancia horizontal ─────────────────────────────────────
    dist_h = np.linalg.norm(p[0] - p[3])   # distancia: comisura izq ↔ comisura der

    # Evitamos división entre cero (si el ojo no se detectó correctamente)
    if dist_h == 0:
        return 0.0

    # Fórmula EAR: promedio vertical / horizontal
    # Se divide entre 2.0 en el denominador porque usamos 2 distancias verticales
    return round((dist_v1 + dist_v2) / (2.0 * dist_h), 3)


def calcular_mar(face_lms):
    """
    Calcula el Mouth Aspect Ratio.

    Parámetro:
        face_lms → lista plana de 468 NormalizedLandmark

    Retorna:
        MAR como float (0.0 a ~1.0)
    """
    # Accedemos a los 4 puntos de la boca usando el diccionario FACE_LANDMARKS
    # Creamos arrays NumPy con [x, y] de cada punto.
    labio_sup = np.array([face_lms[FACE_LANDMARKS['labio_arriba']].x,
                          face_lms[FACE_LANDMARKS['labio_arriba']].y])

    labio_inf = np.array([face_lms[FACE_LANDMARKS['labio_abajo']].x,
                          face_lms[FACE_LANDMARKS['labio_abajo']].y])

    boca_izq  = np.array([face_lms[FACE_LANDMARKS['boca_izquierda']].x,
                          face_lms[FACE_LANDMARKS['boca_izquierda']].y])

    boca_der  = np.array([face_lms[FACE_LANDMARKS['boca_derecha']].x,
                          face_lms[FACE_LANDMARKS['boca_derecha']].y])

    # Distancia vertical: labio superior ↕ labio inferior
    dist_v = np.linalg.norm(labio_sup - labio_inf)

    # Distancia horizontal: comisura izquierda ↔ comisura derecha
    dist_h = np.linalg.norm(boca_izq  - boca_der)

    if dist_h == 0:
        return 0.0

    # Fórmula MAR: no necesita dividir entre 2 porque solo hay 1 distancia vertical
    return round(dist_v / dist_h, 3)


def analizar_expresiones_faciales(imagen_rgb):
    """
    Detecta rostro, calcula EAR y MAR, y determina el estado de ojos y boca.
    """
    # Índices de los 6 puntos que definen cada ojo (estándar para EAR)
    # Orden: [comisura_izq, arriba_1, arriba_2, comisura_der, abajo_2, abajo_1]
    PUNTOS_OJO_IZQ = [33,  160, 158, 133, 153, 144]
    PUNTOS_OJO_DER = [362, 385, 387, 263, 373, 380]

    # Configuración mínima del detector (solo necesitamos landmarks, no blendshapes)
    opciones = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_FACE),
        running_mode=RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
    )

    with FaceLandmarker.create_from_options(opciones) as detector:
        # Detectar landmarks del rostro
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.face_landmarks:
            print("⚠️  No se detectó rostro en la imagen.")
            return

        # resultado.face_landmarks[0] → landmarks del PRIMER rostro detectado
        # (lista plana de 468 NormalizedLandmark)
        face_lms = resultado.face_landmarks[0]

        # ─── Cálculo de métricas ───────────────────────────────────
        ear_izq = calcular_ear(face_lms, PUNTOS_OJO_IZQ)  # EAR ojo izquierdo
        ear_der = calcular_ear(face_lms, PUNTOS_OJO_DER)  # EAR ojo derecho

        # Promediamos los dos ojos para mayor robustez
        # (si un ojo no se detectó bien, el promedio suaviza el error)
        ear_avg = round((ear_izq + ear_der) / 2, 3)

        mar = calcular_mar(face_lms)  # MAR de la boca

    # ─── Imprimir reporte ──────────────────────────────────────────
    print("=" * 50)
    print("  📊 ANÁLISIS DE EXPRESIÓN FACIAL")
    print("=" * 50)
    print(f"  EAR ojo izquierdo : {ear_izq}  (umbral somnolencia: < 0.20)")
    print(f"  EAR ojo derecho   : {ear_der}")
    print(f"  EAR promedio      : {ear_avg}")
    print(f"  MAR boca          : {mar}   (umbral boca abierta: > 0.50)")
    print("─" * 50)

    # ─── Umbrales de decisión ──────────────────────────────────────
    # Operador ternario en Python: valor_si_true if condicion else valor_si_false
    # Equivale a: if ear_avg < 0.20: estado = "cerrados" else: estado = "abiertos"
    estado_ojos = "😴 CERRADOS (posible somnolencia)" if ear_avg < 0.20 else "👀 ABIERTOS"
    estado_boca = "😮 ABIERTA (bostezo/habla)"        if mar     > 0.50  else "😐 CERRADA"

    print(f"  Estado ojos : {estado_ojos}")
    print(f"  Estado boca : {estado_boca}")
    print("=" * 50)

    # ─── Gráfica de barras ────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))

    metricas = ['EAR ojo izq', 'EAR ojo der', 'EAR promedio', 'MAR boca']
    valores  = [ear_izq, ear_der, ear_avg, mar]

    # Color condicionado: verde = normal, rojo = alerta
    colores = ['green' if ear_izq >= 0.20 else 'red',
               'green' if ear_der >= 0.20 else 'red',
               'green' if ear_avg >= 0.20 else 'red',
               'green' if mar     <= 0.50 else 'orange']

    barras = ax.bar(metricas, valores, color=colores, edgecolor='black', alpha=0.8)

    # Líneas de referencia (umbrales)
    ax.axhline(y=0.20, color='red',    linestyle='--', alpha=0.7, label='Umbral EAR (0.20)')
    ax.axhline(y=0.50, color='orange', linestyle='--', alpha=0.7, label='Umbral MAR (0.50)')

    # Etiquetas sobre las barras
    for barra, val in zip(barras, valores):  # zip() combina dos listas elemento a elemento
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.01,
               f'{val}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylim(0, 0.8)
    ax.set_title('EAR y MAR — Análisis de apertura ocular y bucal', fontsize=13)
    ax.set_ylabel('Ratio')
    ax.legend()
    plt.tight_layout()
    plt.show()


print("✅ calcular_ear(), calcular_mar(), analizar_expresiones_faciales() definidas.")
print()
print("💡 EJERCICIO PROPUESTO:")
print("   Sube una foto tuya con los ojos cerrados y observa cómo cambia el EAR.")
print("   Luego sube una foto bostezando y observa el MAR.")
print()
print("   Para usar: analizar_expresiones_faciales(cargar_imagen('tu_foto.jpg'))")


---
# ✋ MÓDULO 2 — Hand Landmarks: 21 Puntos por Mano

## Anatomía de los landmarks de la mano

```
MediaPipe detecta 21 puntos por mano (de base a punta):

  PUNTA:   4    8   12   16   20
           |    |    |    |    |
  MED2:    3    7   11   15   19
           |    |    |    |    |
  MED1:    2    6   10   14   18
           |    |    |    |    |
  BASE:    1    5    9   13   17
            \   |    |    |   /
             \  |    |    |  /
              \ |    |    | /
               \|    |    |/
         0 ←── WRIST (Muñeca)

  DEDO:   Pulgar  Índice  Medio  Anular  Meñique
  BASE:      1      5       9      13      17
  MED1:      2      6      10      14      18
  MED2:      3      7      11      15      19
  PUNTA:     4      8      12      16      20
```

> 💡 **Patrón a memorizar:** Las puntas son 4, 8, 12, 16, 20 (múltiplos de 4 + pulgar 4).
> Las bases de los 4 dedos largos son 5, 9, 13, 17 (múltiplos de 4 + 1).

## ¿Cómo sabe MediaPipe si es mano izquierda o derecha?

El modelo clasifica la lateralidad (handedness) automáticamente.
Los resultados se acceden así:
```python
# Nueva API:
lado = resultado.handedness[i][0].category_name  # → 'Left' o 'Right'

# API legacy:
lado = resultado.multi_handedness[i].classification[0].label
```
Nota: MediaPipe devuelve la mano como si el usuario se mirara en un espejo.
Si la cámara no está invertida, Left y Right pueden aparecer intercambiados.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 2.A — MAPA DE LANDMARKS DE LA MANO                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Definimos el mapa de landmarks con nombres anatómicos en español.
# Los nombres de articulaciones vienen de la anatomía:
#   CMC = carpometacarpal (unión de la mano con la muñeca)
#   MCP = metacarpofalángica (nudillo, unión del dedo con la palma)
#   IP  = interfalángica (articulación a mitad del pulgar)
#   PIP = interfalángica proximal (primer nudo del dedo)
#   DIP = interfalángica distal (segundo nudo del dedo)

HAND_LANDMARKS = {
    # ── MUÑECA ────────────────────────────────────────────────────
    'muneca':          0,   # WRIST — punto central en la base de la mano

    # ── PULGAR (THUMB) ────────────────────────────────────────────
    # El pulgar solo tiene 4 articulaciones (no tiene PIP/DIP como los demás)
    'pulgar_cmc':      1,   # carpometacarpal — donde el pulgar se une a la palma
    'pulgar_mcp':      2,   # metacarpofalángica — primer nudillo del pulgar
    'pulgar_ip':       3,   # interfalángica — única articulación media del pulgar
    'pulgar_punta':    4,   # THUMB_TIP — punta del dedo pulgar

    # ── ÍNDICE (INDEX_FINGER) ─────────────────────────────────────
    'indice_base':     5,   # MCP — base del dedo índice (donde se une a la palma)
    'indice_med1':     6,   # PIP — primera articulación (primer nudo)
    'indice_med2':     7,   # DIP — segunda articulación (segundo nudo)
    'indice_punta':    8,   # INDEX_FINGER_TIP — punta del índice

    # ── MEDIO (MIDDLE_FINGER) ─────────────────────────────────────
    'medio_base':      9,
    'medio_med1':     10,
    'medio_med2':     11,
    'medio_punta':    12,   # MIDDLE_FINGER_TIP

    # ── ANULAR (RING_FINGER) ──────────────────────────────────────
    'anular_base':    13,
    'anular_med1':    14,
    'anular_med2':    15,
    'anular_punta':   16,   # RING_FINGER_TIP

    # ── MEÑIQUE (PINKY_FINGER) ────────────────────────────────────
    'menique_base':   17,
    'menique_med1':   18,
    'menique_med2':   19,
    'menique_punta':  20,   # PINKY_TIP
}

# ─── Diccionarios de acceso rápido ────────────────────────────────
# Para detectar gestos necesitamos acceso rápido a las puntas y bases.
# En lugar de buscar en HAND_LANDMARKS cada vez, creamos atajos.

# Puntas de los 5 dedos (índices 4, 8, 12, 16, 20)
PUNTAS_DEDOS = {
    'pulgar':  4,   # THUMB_TIP
    'indice':  8,   # INDEX_FINGER_TIP
    'medio':  12,   # MIDDLE_FINGER_TIP
    'anular': 16,   # RING_FINGER_TIP
    'menique':20,   # PINKY_TIP
}

# Articulaciones MCP (base/nudillo) de los 5 dedos
# Se usan para comparar la posición de la punta vs la base
# → si la punta está más arriba que la base, el dedo está extendido
BASES_DEDOS = {
    'pulgar':  2,   # MCP del pulgar
    'indice':  5,   # MCP del índice
    'medio':   9,   # MCP del medio
    'anular': 13,   # MCP del anular
    'menique':17,   # MCP del meñique
}

print("✅ HAND_LANDMARKS, PUNTAS_DEDOS y BASES_DEDOS definidos.")
print(f"   Total landmarks: {len(HAND_LANDMARKS)}")
print()
print("📌 PATRÓN DE ÍNDICES:")
print("   Punta del dedo N = 4*N")
print("   Base del dedo N  = 4*N - 3   (exceptuando el pulgar)")
print()
print("   Puntas  → 4 (pulgar), 8 (índice), 12 (medio), 16 (anular), 20 (meñique)")
print("   Bases   → 2 (pulgar), 5 (índice),  9 (medio), 13 (anular), 17 (meñique)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 2.B — DETECCIÓN DE MANOS EN IMAGEN                     ║
# ╚══════════════════════════════════════════════════════════════════╝

def analizar_manos_imagen(imagen_rgb, max_manos=2):
    """
    Detecta y visualiza los 21 landmarks de cada mano en una imagen.

    CAMBIOS API LEGACY → NUEVA API:
        result.multi_hand_landmarks[i]  →  result.hand_landmarks[i]
        result.multi_handedness[i]      →  result.handedness[i]
        handedness.classification[0].label  →  handedness[0].category_name
        mp_hands.HAND_CONNECTIONS       →  HandLandmarksConnections.HAND_CONNECTIONS

    Parámetros:
        imagen_rgb → array NumPy RGB
        max_manos  → máximo de manos a detectar (1 o 2)

    Retorna:
        imagen_anotada → imagen con landmarks dibujados
        resultado      → HandLandmarkerResult
        info_manos     → lista de dicts con datos por mano
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    info_manos     = []  # lista vacía que llenamos con datos de cada mano detectada

    # Configurar el detector de manos
    opciones = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_HAND),
        running_mode=RunningMode.IMAGE,

        # num_hands: detectará hasta 2 manos simultáneamente.
        # Si solo necesitas 1, cambia a num_hands=1 para mayor velocidad.
        num_hands=max_manos,

        # Umbral para iniciar el seguimiento de una mano nueva
        min_hand_detection_confidence=0.5,

        # Umbral para confirmar que una mano sigue presente en el frame
        min_hand_presence_confidence=0.5,

        # Umbral para el seguimiento entre frames (VIDEO/LIVE_STREAM)
        min_tracking_confidence=0.5,
    )

    with HandLandmarker.create_from_options(opciones) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.hand_landmarks:
            print("⚠️  No se detectaron manos en la imagen.")
            return imagen_anotada, None, []

        print(f"✅ {len(resultado.hand_landmarks)} mano(s) detectada(s).")

        # ── Iterar sobre cada mano ─────────────────────────────────
        # resultado.hand_landmarks[i] y resultado.handedness[i] son listas PARALELAS:
        # el índice i corresponde a la MISMA mano en ambas listas.
        for idx_mano in range(len(resultado.hand_landmarks)):

            # Landmarks de la mano i: lista plana de 21 NormalizedLandmark
            hand_lms = resultado.hand_landmarks[idx_mano]

            # Clasificación de la mano: lista de Category con .category_name
            # handedness[0] es el resultado de mayor confianza
            handedness = resultado.handedness[idx_mano]

            # .category_name devuelve 'Left' o 'Right'
            # En la API legacy era: handedness.classification[0].label
            etiqueta  = handedness[0].category_name   # 'Left' o 'Right'
            confianza = handedness[0].score            # 0.0 a 1.0

            print(f"   Mano {idx_mano+1}: {etiqueta} ({confianza:.1%} confianza)")
            # :.1% formatea como porcentaje con 1 decimal (ej: 0.987 → 98.7%)

            # ── Dibujar esqueleto de la mano ───────────────────────
            mp_drawing.draw_landmarks(
                image=imagen_anotada,
                landmark_list=hand_lms,
                # HandLandmarksConnections.HAND_CONNECTIONS define qué puntos conectar:
                # muñeca-base-med1-med2-punta para cada dedo
                # En la API legacy: mp_hands.HAND_CONNECTIONS
                connections=HandLandmarksConnections.HAND_CONNECTIONS,
                # Estilos predefinidos: colores diferentes por articulación
                landmark_drawing_spec=mp_drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=mp_drawing_styles.get_default_hand_connections_style(),
            )

            # ── Resaltar las puntas de los 5 dedos ─────────────────
            for nombre_dedo, idx_lm in PUNTAS_DEDOS.items():
                lm = hand_lms[idx_lm]  # acceso directo: hand_lms[índice]
                px, py = landmark_a_pixeles(lm, alto, ancho)

                # Círculo exterior blanco (borde)
                cv2.circle(imagen_anotada, (px, py), 10, (255, 255, 255), 2)
                # Círculo interior de color
                cv2.circle(imagen_anotada, (px, py),  6, (0, 200, 255), -1)
                # Etiqueta con el nombre del dedo (primeras 4 letras)
                cv2.putText(imagen_anotada, nombre_dedo[:4], (px-10, py-12),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)

            # ── Numerar los 21 landmarks ───────────────────────────
            # enumerate(hand_lms) devuelve (índice, landmark) para cada punto
            for j, lm in enumerate(hand_lms):
                px, py = landmark_a_pixeles(lm, alto, ancho)
                cv2.putText(imagen_anotada, str(j), (px+4, py-4),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)
                # str(j) convierte el entero j a string '0', '1', '2'...

            # ── Tabla de coordenadas ───────────────────────────────
            # Comprensión de lista: crea una fila por cada landmark de HAND_LANDMARKS
            filas = [
                {'Mano': etiqueta, 'Punto': nombre, 'Índice': idx_lm,
                 'x (norm)': round(hand_lms[idx_lm].x, 4),
                 'y (norm)': round(hand_lms[idx_lm].y, 4),
                 'z (prof)': round(hand_lms[idx_lm].z, 4),
                 'x (px)': landmark_a_pixeles(hand_lms[idx_lm], alto, ancho)[0],
                 'y (px)': landmark_a_pixeles(hand_lms[idx_lm], alto, ancho)[1]}
                for nombre, idx_lm in HAND_LANDMARKS.items()
            ]

            # Empaquetamos toda la información de esta mano en un diccionario
            info_manos.append({
                'etiqueta':  etiqueta,
                'confianza': confianza,
                'landmarks': hand_lms,
                'tabla':     pd.DataFrame(filas),
            })

    return imagen_anotada, resultado, info_manos


# ─── Widget de análisis de manos ──────────────────────────────────
uploader_mano     = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='✋ Subir foto')
btn_analizar_mano = widgets.Button(description='Analizar Mano', button_style='info')
salida_mano       = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen con una o dos manos (fondo limpio funciona mejor):</b>'),
    uploader_mano, btn_analizar_mano, salida_mano
]))


def on_analizar_mano(_):
    with salida_mano:
        clear_output()
        if not uploader_mano.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_mano.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))

        img_anotada, resultado, info = analizar_manos_imagen(img_rgb, max_manos=2)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Hand Landmarks — 21 puntos/mano')

        for mano in info:
            print(f"\n📊 Coordenadas — Mano {mano['etiqueta']}:")
            display(mano['tabla'].head(10))  # .head(10) muestra solo las primeras 10 filas


btn_analizar_mano.on_click(on_analizar_mano)
print("✅ analizar_manos_imagen() definida y widget listo.")


---
# 🧍 MÓDULO 3 — Body Pose: 33 Landmarks del Cuerpo

## Mapa del cuerpo humano

```
         0 (nariz)
    1  2  3  4   ← cara: ojo_izq (1-3), ojo_der (4-6)
       7   8     ← orejas
       9  10     ← boca
      11  12     ← hombros
      13  14     ← codos
      15  16     ← muñecas
   17  18  19  20  21  22  ← dedos
      23  24     ← caderas
      25  26     ← rodillas
      27  28     ← tobillos
      29  30     ← talones
      31  32     ← puntas de pies

REGLA: Impar = izquierda del sujeto | Par = derecha del sujeto
```

## ¿Qué mide .visibility?

A diferencia de los landmarks de mano/rostro, los de cuerpo tienen un
atributo extra: `.visibility` (0.0 a 1.0).

Indica qué tan visible está el punto:
- `visibility = 1.0` → punto completamente visible y confiable
- `visibility = 0.5` → parcialmente visible (ej: detrás de un objeto)
- `visibility = 0.0` → punto no visible (detrás del cuerpo, fuera de cuadro)

**Buena práctica:** antes de usar un landmark del cuerpo, verifica:
```python
if landmarks[idx].visibility > 0.5:
    # el punto es confiable, úsalo
```


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 3.A — MAPA DE LANDMARKS DEL CUERPO                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Los 33 landmarks del cuerpo con nombres descriptivos en español.
# La convención de MediaPipe es: impar = izquierda, par = derecha
# (desde el punto de vista del sujeto, no de la cámara).

POSE_LANDMARKS = {
    # ── CABEZA / CARA ──────────────────────────────────────────────
    'nariz':            0,  # punta de la nariz — punto de referencia central
    'ojo_izq_inner':    1,  # esquina interna (nasal) del ojo izquierdo del sujeto
    'ojo_izq':          2,  # centro del ojo izquierdo
    'ojo_izq_outer':    3,  # esquina externa (temporal) del ojo izquierdo
    'ojo_der_inner':    4,
    'ojo_der':          5,
    'ojo_der_outer':    6,
    'oreja_izq':        7,  # oreja izquierda del sujeto
    'oreja_der':        8,
    'boca_izq':         9,  # comisura izquierda de la boca
    'boca_der':        10,

    # ── TREN SUPERIOR ─────────────────────────────────────────────
    'hombro_izq':      11,  # articulación del hombro izquierdo
    'hombro_der':      12,
    'codo_izq':        13,  # codo izquierdo
    'codo_der':        14,
    'muneca_izq':      15,  # muñeca izquierda
    'muneca_der':      16,
    'pulgar_izq':      17,  # base del pulgar izquierdo
    'pulgar_der':      18,
    'indice_izq':      19,  # punta del dedo índice izquierdo
    'indice_der':      20,
    'menique_izq':     21,  # punta del meñique izquierdo
    'menique_der':     22,

    # ── TREN INFERIOR ─────────────────────────────────────────────
    'cadera_izq':      23,  # articulación de la cadera izquierda
    'cadera_der':      24,
    'rodilla_izq':     25,  # rodilla izquierda
    'rodilla_der':     26,
    'tobillo_izq':     27,  # tobillo izquierdo
    'tobillo_der':     28,
    'talon_izq':       29,  # talón izquierdo
    'talon_der':       30,
    'pie_izq':         31,  # punta del pie izquierdo
    'pie_der':         32,
}

# ─── Ángulos biomecánicos ─────────────────────────────────────────
# Definimos los ángulos que queremos medir como tuplas (A, B, C):
# A-B-C donde B es el VÉRTICE (la articulación donde se mide el ángulo).
# Por ejemplo: 'codo_izquierdo': (11, 13, 15)
#   → A=hombro(11), B=codo(13), C=muñeca(15)
#   → medimos el ángulo de flexión del codo izquierdo

ANGULOS_BIOMECÁNICOS = {
    'codo_izquierdo':    (11, 13, 15),  # hombro_izq - codo_izq - muneca_izq
    'codo_derecho':      (12, 14, 16),  # hombro_der - codo_der - muneca_der
    'hombro_izquierdo':  (13, 11, 23),  # codo_izq   - hombro_izq - cadera_izq
    'hombro_derecho':    (14, 12, 24),
    'cadera_izquierda':  (11, 23, 25),  # hombro_izq - cadera_izq - rodilla_izq
    'cadera_derecha':    (12, 24, 26),
    'rodilla_izquierda': (23, 25, 27),  # cadera_izq - rodilla_izq - tobillo_izq
    'rodilla_derecha':   (24, 26, 28),
    'tobillo_izquierdo': (25, 27, 31),  # rodilla_izq - tobillo_izq - pie_izq
    'tobillo_derecho':   (26, 28, 32),
}

print("✅ POSE_LANDMARKS y ANGULOS_BIOMECÁNICOS definidos.")
print(f"   Total landmarks: {len(POSE_LANDMARKS)}")
print(f"   Ángulos biomecánicos: {len(ANGULOS_BIOMECÁNICOS)}")
print()
print("📐 ÁNGULOS DISPONIBLES:")
for nombre, (a, b, c) in ANGULOS_BIOMECÁNICOS.items():
    print(f"   {nombre:<22} → vértice landmark {b:>2} entre landmarks {a} y {c}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 3.B — DETECCIÓN DE POSE Y ÁNGULOS ARTICULARES          ║
# ╚══════════════════════════════════════════════════════════════════╝

def analizar_pose_imagen(imagen_rgb, calcular_angulos=True, modelo='full'):
    """
    Detecta los 33 landmarks del cuerpo y calcula ángulos articulares.

    CAMBIOS API LEGACY → NUEVA API:
        mp_pose.Pose(model_complexity=0/1/2)   →  PoseLandmarkerOptions(base_options=MODEL_POSE_LITE/FULL/HEAVY)
        result.pose_landmarks.landmark[i]       →  result.pose_landmarks[0][i]
        mp_pose.POSE_CONNECTIONS                →  PoseLandmarksConnections.POSE_LANDMARKS

    Parámetros:
        imagen_rgb       → array NumPy RGB
        calcular_angulos → si True, calcula y muestra ángulos de articulaciones visibles
        modelo           → 'lite' (rápido), 'full' (balanceado), 'heavy' (preciso)
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    angulos_dict   = {}  # diccionario para guardar los ángulos calculados

    # ── Selección del modelo según parámetro ──────────────────────
    # dict.get(key, default) devuelve el valor para 'key', o 'default' si no existe.
    # Esto previene errores si el usuario escribe 'Fast' en lugar de 'lite'.
    modelo_path = {
        'lite':  MODEL_POSE_LITE,
        'full':  MODEL_POSE_FULL,
        'heavy': MODEL_POSE_HEAVY,
    }.get(modelo, MODEL_POSE_FULL)  # por defecto usa 'full' si el modelo no se reconoce

    opciones = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=modelo_path),
        running_mode=RunningMode.IMAGE,
        num_poses=1,  # detecta solo 1 persona (máximo soportado en modo IMAGE)
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        # output_segmentation_masks=True genera una máscara que separa la persona
        # del fondo. Útil para efectos visuales. Se desactiva aquí por velocidad.
        output_segmentation_masks=False,
    )

    with PoseLandmarker.create_from_options(opciones) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        # resultado.pose_landmarks es una lista de listas.
        # resultado.pose_landmarks[0]    → landmarks de la PRIMERA persona
        # resultado.pose_landmarks[0][i] → landmark i de la primera persona
        # Cada landmark tiene .x, .y, .z, .visibility
        if not resultado.pose_landmarks:
            print("⚠️  No se detectó ninguna persona en la imagen.")
            return imagen_anotada, {}, pd.DataFrame()

        print(f"✅ Persona detectada (modelo: {modelo}).")

        # Tomamos los landmarks de la primera (y única) persona detectada
        landmarks = resultado.pose_landmarks[0]  # lista plana de 33 NormalizedLandmark

        # ── Dibujar el esqueleto ───────────────────────────────────
        mp_drawing.draw_landmarks(
            image=imagen_anotada,
            landmark_list=landmarks,
            # PoseLandmarksConnections.POSE_LANDMARKS define las conexiones:
            # tobillo-rodilla-cadera-hombro-codo-muñeca, etc.
            # En la API legacy: mp_pose.POSE_CONNECTIONS
            connections=PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(255, 255, 0),  # amarillo para las líneas del esqueleto
                thickness=3
            ),
        )

        # ── Calcular ángulos articulares ───────────────────────────
        if calcular_angulos:
            for nombre, (idx_a, idx_b, idx_c) in ANGULOS_BIOMECÁNICOS.items():
                lm_a = landmarks[idx_a]  # acceso directo por índice
                lm_b = landmarks[idx_b]  # vértice de la articulación
                lm_c = landmarks[idx_c]

                # Solo calculamos si los 3 puntos son visibles.
                # .visibility > 0.5 garantiza que el punto es confiable.
                if (lm_a.visibility > 0.5 and lm_b.visibility > 0.5
                        and lm_c.visibility > 0.5):

                    # calcular_angulo() trabaja con listas [x, y]
                    # Le pasamos las coordenadas normalizadas de los 3 puntos
                    angulo = calcular_angulo(
                        [lm_a.x, lm_a.y],
                        [lm_b.x, lm_b.y],
                        [lm_c.x, lm_c.y]
                    )
                    angulos_dict[nombre] = angulo  # guardamos en el diccionario

                    # ── Mostrar el ángulo sobre la imagen ──────────
                    # Posición del texto: sobre el vértice B (la articulación)
                    px_b, py_b = landmark_a_pixeles(lm_b, alto, ancho)
                    texto = f"{angulo:.0f}°"  # :.0f = sin decimales

                    # Calcular el tamaño del texto para centrar el fondo negro
                    # cv2.getTextSize devuelve ((ancho, alto), baseline)
                    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)

                    # Dibujamos un rectángulo negro detrás del texto (mejor legibilidad)
                    cv2.rectangle(imagen_anotada,
                                 (px_b-5, py_b-th-10), (px_b+tw+5, py_b+5),
                                 (0, 0, 0), -1)  # -1 = relleno sólido

                    # Color del texto: verde si el ángulo es normal (30°-170°),
                    # rojo si es extremo (articulación muy forzada)
                    color_texto = (0, 255, 0) if 30 < angulo < 170 else (0, 50, 255)

                    cv2.putText(imagen_anotada, texto, (px_b, py_b-5),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)

        # ── Tabla de coordenadas ───────────────────────────────────
        filas = [{
            'Punto': nombre,
            'Índice': idx,
            'x (norm)': round(landmarks[idx].x, 4),
            'y (norm)': round(landmarks[idx].y, 4),
            'z (prof)': round(landmarks[idx].z, 4),
            'visibilidad': round(landmarks[idx].visibility, 3),
            'x (px)': landmark_a_pixeles(landmarks[idx], alto, ancho)[0],
            'y (px)': landmark_a_pixeles(landmarks[idx], alto, ancho)[1]
        } for nombre, idx in POSE_LANDMARKS.items()]

        tabla_coords = pd.DataFrame(filas)

    return imagen_anotada, angulos_dict, tabla_coords


def visualizar_angulos(angulos_dict):
    """Muestra los ángulos articulares como barras horizontales codificadas por color."""
    if not angulos_dict:
        print("⚠️  No hay ángulos para visualizar.")
        return

    nombres = list(angulos_dict.keys())
    valores = list(angulos_dict.values())

    # Color según el tipo de articulación
    colores = ['#2196F3' if 'codo' in n else
               '#4CAF50' if 'rodilla' in n else
               '#FF9800' if 'hombro' in n else
               '#9C27B0' if 'cadera' in n else
               '#F44336' for n in nombres]

    fig, ax = plt.subplots(figsize=(12, 5))
    barras = ax.barh(nombres, valores, color=colores, edgecolor='white', alpha=0.85)

    # Etiquetas con el valor del ángulo al final de cada barra
    for barra, val in zip(barras, valores):
        ax.text(val+1, barra.get_y()+barra.get_height()/2,
               f'{val:.1f}°', va='center', fontsize=10, fontweight='bold')

    # Líneas de referencia
    ax.axvline(x=180, color='gray', linestyle='--', alpha=0.5, label='180° (completamente extendido)')
    ax.axvline(x=90,  color='blue', linestyle='--', alpha=0.5, label='90° (ángulo recto)')

    ax.set_xlim(0, 200)
    ax.set_xlabel('Ángulo en grados', fontsize=11)
    ax.set_title('📐 Ángulos Articulares Biomecánicos', fontsize=14, fontweight='bold')

    # Leyenda de colores
    leyenda = [mpatches.Patch(color=c, label=l) for c, l in
               [('#2196F3','Codo'), ('#4CAF50','Rodilla'), ('#FF9800','Hombro'), ('#9C27B0','Cadera')]]
    ax.legend(handles=leyenda, loc='lower right')
    plt.tight_layout()
    plt.show()


# ─── Widget de análisis de pose ───────────────────────────────────
uploader_pose     = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🧍 Subir foto')
btn_analizar_pose = widgets.Button(description='Analizar Postura', button_style='warning')
salida_pose       = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen de cuerpo completo o medio cuerpo:</b>'),
    uploader_pose, btn_analizar_pose, salida_pose
]))


def on_analizar_pose(_):
    with salida_pose:
        clear_output()
        if not uploader_pose.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_pose.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))

        img_anotada, angulos, tabla = analizar_pose_imagen(img_rgb, calcular_angulos=True, modelo='full')
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Pose — 33 landmarks + ángulos')

        if angulos:
            print("\n📐 Ángulos articulares detectados:")
            for nombre, grados in angulos.items():
                print(f"   {nombre:<25}: {grados}°")
            visualizar_angulos(angulos)

        if not tabla.empty:
            print("\n📊 Coordenadas de los 33 landmarks:")
            display(tabla)


btn_analizar_pose.on_click(on_analizar_pose)
print("✅ analizar_pose_imagen() y visualizar_angulos() definidas.")


---
# 🔮 MÓDULO 4 — Holistic: Rostro + Manos + Cuerpo Simultáneo

## ¿Por qué necesitamos Holistic?

Ejecutar los 3 detectores por separado es ineficiente.
Holistic los combina para analizar todo el cuerpo de una vez:

```
UNA imagen → 3 detectores → obtienes:
  ✅ 468 landmarks del rostro
  ✅  33 landmarks del cuerpo
  ✅  21 landmarks mano izquierda
  ✅  21 landmarks mano derecha
  ─────────────────────────────────
  TOTAL: 543 puntos simultáneos
```

> ⚠️ **Nota:** La nueva API mp.tasks NO incluye Holistic como módulo único.
> Aquí lo reimplementamos ejecutando los 3 detectores en secuencia
> sobre la misma imagen, con resultados visuales idénticos.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 4 — HOLISTIC: ROSTRO + MANOS + CUERPO                  ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── ¿Por qué no existe Holistic en la nueva API? ─────────────────
# La API LEGACY tenía mp.solutions.holistic.Holistic, que combinaba
# los 3 modelos en UNA SOLA pasada (más eficiente en pipeline).
# La nueva API mp.tasks separó los modelos para mayor flexibilidad.
# La solución: ejecutar los 3 detectores EN SECUENCIA sobre la misma imagen.
# Tiempo extra ≈ 50-100ms por imagen (aceptable para imágenes estáticas).

def analizar_holistic_imagen(imagen_rgb):
    """
    Ejecuta detección simultánea de rostro, cuerpo y manos.

    Arquitectura:
        1. Convertir imagen UNA VEZ (se reutiliza en los 3 detectores)
        2. Detector de rostro  → 468 landmarks
        3. Detector de cuerpo  → 33 landmarks
        4. Detector de manos   → 21 landmarks × 2 manos
        5. Combinar resultados en imagen única

    Retorna:
        imagen_anotada → imagen con todos los landmarks
        resumen        → dict {'rostro': N, 'cuerpo': N, 'mano_izq': N, 'mano_der': N}
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    resumen        = {'rostro': 0, 'cuerpo': 0, 'mano_izq': 0, 'mano_der': 0}

    # ── Conversión única de imagen ────────────────────────────────
    # OPTIMIZACIÓN: convertimos la imagen a mp.Image UNA SOLA VEZ
    # y la reutilizamos en los 3 detectores.
    # Si la convirtiéramos dentro de cada with-block, haríamos la operación 3 veces.
    mp_imagen = rgb_a_mp_image(imagen_rgb)

    # ── DETECTOR 1: Rostro ────────────────────────────────────────
    # Cada detector se crea, usa y cierra en su propio bloque 'with'.
    # Esto libera recursos inmediatamente al salir del bloque.
    opts_face = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_FACE),
        running_mode=RunningMode.IMAGE,
        num_faces=1,
    )
    with FaceLandmarker.create_from_options(opts_face) as det_face:
        res_face = det_face.detect(mp_imagen)

    # Los resultados (res_face) se guardan FUERA del bloque with
    # y persisten después de que el detector se cierra. Los datos están en memoria.
    if res_face.face_landmarks:
        resumen['rostro'] = 468
        face_lms = res_face.face_landmarks[0]

        # Dibuja la malla facial en verde oscuro
        mp_drawing.draw_landmarks(
            imagen_anotada, face_lms,
            FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(80, 110, 10), thickness=1, circle_radius=1
            ),
        )
        # Dibuja el contorno en verde brillante
        mp_drawing.draw_landmarks(
            imagen_anotada, face_lms,
            FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1),
        )

    # ── DETECTOR 2: Cuerpo (Pose) ─────────────────────────────────
    opts_pose = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_POSE_FULL),
        running_mode=RunningMode.IMAGE,
        num_poses=1,
    )
    with PoseLandmarker.create_from_options(opts_pose) as det_pose:
        res_pose = det_pose.detect(mp_imagen)

    if res_pose.pose_landmarks:
        resumen['cuerpo'] = 33
        mp_drawing.draw_landmarks(
            imagen_anotada, res_pose.pose_landmarks[0],
            PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=3),
        )

    # ── DETECTOR 3: Manos ─────────────────────────────────────────
    opts_hand = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_HAND),
        running_mode=RunningMode.IMAGE,
        num_hands=2,
    )
    with HandLandmarker.create_from_options(opts_hand) as det_hand:
        res_hand = det_hand.detect(mp_imagen)

    if res_hand.hand_landmarks:
        # zip() combina dos listas en pares: (landmarks[0], handedness[0]), etc.
        for hand_lms, handedness in zip(res_hand.hand_landmarks, res_hand.handedness):
            lado = handedness[0].category_name  # 'Left' o 'Right'
            # Colores diferentes para cada mano
            color_mano = (0, 200, 255) if lado == 'Left' else (255, 100, 200)

            if lado == 'Left':
                resumen['mano_izq'] = 21
            else:
                resumen['mano_der'] = 21

            mp_drawing.draw_landmarks(
                imagen_anotada, hand_lms,
                HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=color_mano, thickness=2),
            )

    # ── Leyenda sobre la imagen ───────────────────────────────────
    # Dibujamos un panel de información en la esquina superior izquierda
    leyenda_items = [
        (f"Rostro: {resumen['rostro']} pts",   (80, 255, 80)),
        (f"Cuerpo: {resumen['cuerpo']} pts",   (245, 117, 66)),
        (f"M.Izq:  {resumen['mano_izq']} pts", (0, 200, 255)),
        (f"M.Der:  {resumen['mano_der']} pts", (255, 100, 200)),
    ]
    for i, (texto, color) in enumerate(leyenda_items):
        # Fondo negro para cada línea de la leyenda
        cv2.rectangle(imagen_anotada, (10, 10+i*30), (200, 35+i*30), (0, 0, 0), -1)
        cv2.putText(imagen_anotada, texto, (15, 28+i*30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

    return imagen_anotada, resumen


# ─── Widget ───────────────────────────────────────────────────────
uploader_holistic = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🔮 Subir imagen')
btn_holistic      = widgets.Button(description='Análisis Completo', button_style='danger')
salida_holistic   = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen que muestre rostro + cuerpo + manos visibles:</b>'),
    uploader_holistic, btn_holistic, salida_holistic
]))


def on_holistic(_):
    with salida_holistic:
        clear_output()
        if not uploader_holistic.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_holistic.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))

        print("🔄 Procesando con 3 detectores: rostro + cuerpo + manos...")
        img_anotada, resumen = analizar_holistic_imagen(img_rgb)

        total = sum(resumen.values())
        print(f"✅ Total de landmarks detectados: {total}")
        for zona, count in resumen.items():
            print(f"   {'✅' if count > 0 else '❌'} {zona:<12}: {count} puntos")
        mostrar_comparacion(img_rgb, img_anotada, 'Original', f'Holistic — {total} landmarks')


btn_holistic.on_click(on_holistic)
print("✅ analizar_holistic_imagen() definida.")


---
# ✊ MÓDULO 5 — Detección de Gestos con la Mano

## ¿Cómo detectamos gestos?

Usamos **geometría simple**: comparamos la posición de la PUNTA del dedo
con su BASE. Si la punta está más arriba (y menor en coordenadas de imagen),
el dedo está extendido.

```
Coordenadas de imagen:  y=0 está ARRIBA, y=1 está ABAJO

Dedo EXTENDIDO:   punta.y < base.y   (punta más arriba)
Dedo DOBLADO:     punta.y > base.y   (punta más abajo)

Pulgar (caso especial): usa el eje X, no Y,
porque el pulgar se extiende horizontalmente.
```

Además, usamos **GestureRecognizer** de MediaPipe que reconoce gestos con ML:

| Gesto ML | Emoji | Descripción |
|---|---|---|
| Closed_Fist | ✊ | Puño cerrado |
| Open_Palm | ✋ | Mano abierta |
| Pointing_Up | ☝️ | Apuntando arriba |
| Thumb_Up | 👍 | Pulgar arriba |
| Thumb_Down | 👎 | Pulgar abajo |
| Victory | ✌️ | Señal de paz/victoria |
| ILoveYou | 🤟 | Te amo (lenguaje de señas) |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 5 — DETECCIÓN DE GESTOS (ML + Lógica Geométrica)       ║
# ╚══════════════════════════════════════════════════════════════════╝

# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 1: detectar_dedos_extendidos()
# Algoritmo geométrico (no ML) — funciona con o sin GestureRecognizer
# ════════════════════════════════════════════════════════════════════

def detectar_dedos_extendidos(hand_lms):
    """
    Determina cuáles de los 5 dedos están extendidos.
    Algoritmo: compara posición (y) de la punta vs la base del dedo.

    Recibe hand_lms: lista plana de 21 NormalizedLandmark
    Retorna: dict {'pulgar': bool, 'indice': bool, 'medio': bool, 'anular': bool, 'menique': bool}
    """
    # Extraemos los objetos de las puntas y bases usando PUNTAS_DEDOS y BASES_DEDOS
    puntas = {dedo: hand_lms[idx] for dedo, idx in PUNTAS_DEDOS.items()}
    bases  = {dedo: hand_lms[idx] for dedo, idx in BASES_DEDOS.items()}
    # Resultado: {'pulgar': landmark_4, 'indice': landmark_8, ...}

    dedos_ext = {}  # diccionario de resultados: True = extendido, False = doblado

    # ── Pulgar: eje X ─────────────────────────────────────────────
    # El pulgar es diferente: se extiende horizontalmente.
    # Si la punta está más LEJOS de la muñeca (en X) que la base → extendido.
    pulgar_punta = hand_lms[4]   # THUMB_TIP
    pulgar_base  = hand_lms[2]   # THUMB_MCP
    muneca       = hand_lms[0]   # WRIST — punto de referencia

    # abs() calcula el valor absoluto (distancia sin signo)
    # Si la punta está más lejos de la muñeca que la base en X → extendido
    dedos_ext['pulgar'] = abs(pulgar_punta.x - muneca.x) > abs(pulgar_base.x - muneca.x)

    # ── Los otros 4 dedos: eje Y ──────────────────────────────────
    # RECORDATORIO: y=0 es ARRIBA, y=1 es ABAJO en coordenadas de imagen
    # Si la punta está más ARRIBA (y menor) que la base → dedo extendido
    for dedo in ['indice', 'medio', 'anular', 'menique']:
        # puntas[dedo].y = coordenada y de la punta del dedo
        # bases[dedo].y  = coordenada y de la base del dedo
        # Si punta < base → punta está más arriba → dedo extendido
        dedos_ext[dedo] = puntas[dedo].y < bases[dedo].y

    return dedos_ext  # {'pulgar': True/False, 'indice': True/False, ...}


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 2: clasificar_gesto()
# Reglas manuales para gestos comunes
# ════════════════════════════════════════════════════════════════════

def clasificar_gesto(dedos):
    """
    Clasifica el gesto según el patrón de dedos extendidos.

    Recibe: dict de {'pulgar': bool, 'indice': bool, 'medio': bool, 'anular': bool, 'menique': bool}
    Retorna: tupla (emoji, nombre_gesto)
    """
    # Desempaquetamos el diccionario en variables para mayor claridad
    p  = dedos['pulgar']   # pulgar
    i  = dedos['indice']   # índice
    m  = dedos['medio']    # medio
    a  = dedos['anular']   # anular
    me = dedos['menique']  # meñique

    # ─── Tabla de patrones ─────────────────────────────────────────
    # not any([...]) = ninguno está extendido
    # all([...])     = todos están extendidos
    if not any([p, i, m, a, me]):                      return '✊', 'Puño cerrado'
    if all([p, i, m, a, me]):                           return '✋', 'Mano abierta'
    if p and not i and not m and not a and not me:      return '👍', 'Pulgar arriba'
    if not p and i and not m and not a and not me:      return '☝️', 'Apuntando'
    if not p and i and m and not a and not me:          return '✌️', 'Victoria/Paz'
    if not p and i and m and a and not me:              return '🤟', 'Tres dedos'
    if not p and i and m and a and me:                  return '🖖', 'Cuatro dedos'
    if p and not i and not m and not a and me:          return '🤙', 'Llámame'
    if p and i and not m and not a and not me:          return '🤘', 'Rock'

    # Si ningún patrón coincide, informamos cuántos dedos están extendidos
    # sum() sobre bools cuenta cuántos son True
    return '❓', f'{sum([p, i, m, a, me])} dedos extendidos'


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN 3: analizar_gestos_imagen()
# Usa GestureRecognizer de MediaPipe (ML) + lógica geométrica (comparación)
# ════════════════════════════════════════════════════════════════════

def analizar_gestos_imagen(imagen_rgb):
    """
    Detecta manos y reconoce gestos usando GestureRecognizer.

    GestureRecognizer es HandLandmarker + clasificador de gestos en uno.
    Campos adicionales vs HandLandmarker:
        resultado.gestures[i][0].category_name → nombre del gesto ML
        resultado.gestures[i][0].score          → confianza del gesto
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]

    # GestureRecognizerOptions combina detección de manos + clasificación de gestos
    opts = GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_GESTURE),
        running_mode=RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.6,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    with GestureRecognizer.create_from_options(opts) as recognizer:
        # GestureRecognizer usa .recognize() (no .detect())
        resultado = recognizer.recognize(rgb_a_mp_image(imagen_rgb))

        if not resultado.hand_landmarks:
            print("⚠️  No se detectaron manos.")
            return imagen_anotada

        print("=" * 55)
        print("  🤚 ANÁLISIS DE GESTOS")
        print("=" * 55)

        for idx in range(len(resultado.hand_landmarks)):
            hand_lms   = resultado.hand_landmarks[idx]   # lista de 21 landmarks
            handedness = resultado.handedness[idx]        # lateralidad
            etiqueta   = handedness[0].category_name      # 'Left' o 'Right'

            # ── Gesto reconocido por ML ────────────────────────────
            # resultado.gestures[i] es una lista de Category ordenada por confianza.
            # [0] = gesto más probable
            gesto_ml        = resultado.gestures[idx][0].category_name  # 'Thumb_Up', etc.
            confianza_gesto = resultado.gestures[idx][0].score           # 0.0 a 1.0

            # ── Gesto por lógica geométrica (para comparar con ML) ─
            dedos          = detectar_dedos_extendidos(hand_lms)
            emoji_geo, nombre_geo = clasificar_gesto(dedos)

            print(f"  Mano {etiqueta}:")
            print(f"    🤖 Gesto ML        : {gesto_ml} ({confianza_gesto:.1%})")
            print(f"    📐 Gesto geométrico: {emoji_geo} {nombre_geo}")
            print(f"    Dedos extendidos: ", end='')
            for dedo, ext in dedos.items():
                # '↑' si extendido, '↓' si doblado
                print(f"{dedo[:3]}{'↑' if ext else '↓'}", end=' ')
            print()

            # Dibuja el esqueleto de la mano
            mp_drawing.draw_landmarks(
                imagen_anotada, hand_lms,
                HandLandmarksConnections.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style(),
            )

            # Muestra el gesto ML sobre la imagen
            muneca = hand_lms[0]
            px_m = int(muneca.x * ancho)
            py_m = int(muneca.y * alto)
            texto_gesto = f"{gesto_ml} ({confianza_gesto:.0%})"
            (tw, th), _ = cv2.getTextSize(texto_gesto, cv2.FONT_HERSHEY_DUPLEX, 0.7, 2)
            cv2.rectangle(imagen_anotada, (px_m-5, py_m+10), (px_m+tw+10, py_m+th+25), (0, 0, 0), -1)
            cv2.putText(imagen_anotada, texto_gesto, (px_m, py_m+th+15),
                       cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 255, 200), 2)

        print("=" * 55)

    return imagen_anotada


# ─── Widget ───────────────────────────────────────────────────────
uploader_gesto = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='✊ Subir gesto')
btn_gesto      = widgets.Button(description='Detectar Gesto', button_style='info')
salida_gesto   = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una foto haciendo un gesto con la mano (fondo limpio funciona mejor):</b>'),
    uploader_gesto, btn_gesto, salida_gesto
]))


def on_gesto(_):
    with salida_gesto:
        clear_output()
        if not uploader_gesto.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_gesto.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada = analizar_gestos_imagen(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Gesto detectado')


btn_gesto.on_click(on_gesto)
print("✅ Módulo 5 listo: detectar_dedos_extendidos(), clasificar_gesto(), analizar_gestos_imagen()")


---
# 🚨 MÓDULO 6 — Detección de Caídas y Posturas Incorrectas

## ¿Cómo se detectan?

Analizamos las **relaciones geométricas** entre landmarks del cuerpo:

```
CAÍDA:
  La diferencia (y_tobillo - y_cadera) es pequeña
  → persona horizontal (cadera casi al mismo nivel que los pies)

ESPALDA ENCORVADA:
  El ángulo hombro-cadera-rodilla < 150°
  → la columna no está recta

ASIMETRÍA DE HOMBROS:
  La diferencia de altura (y) entre hombro izq y der > 15px
  → postura inclinada lateralmente
```


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 6 — DETECCIÓN DE CAÍDAS Y POSTURAS INCORRECTAS         ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── Umbrales de detección ────────────────────────────────────────
# Los umbrales son los LÍMITES que definen cuándo algo es 'anormal'.
# Los definimos como diccionario para poder ajustarlos fácilmente
# sin buscar en el código dónde están los números.
# BUENA PRÁCTICA: nunca escribas números mágicos en el código;
# ponlos en variables con nombres descriptivos.

UMBRALES_POSTURA = {
    # Si (y_tobillo - y_cadera) < este valor → posible caída
    # (la persona está casi horizontal)
    'caida_diferencia_y':  0.15,  # en coordenadas normalizadas (0.0-1.0)

    # Si el ángulo del tronco con la vertical > este valor → caída
    'caida_angulo_tronco': 45,    # grados

    # Si el ángulo hombro-cadera-rodilla < este valor → espalda encorvada
    'espalda_angulo_min':  150,   # grados (una espalda recta es ≈ 170-180°)

    # Diferencia máxima permitida entre alturas de hombros
    'hombros_nivel_max':   15,    # píxeles
}


def calcular_angulo_tronco(landmarks_np):
    """
    Calcula el ángulo entre el tronco y la vertical.
    0° = persona completamente erguida
    90° = persona completamente horizontal (acostada)

    Parámetro:
        landmarks_np → dict {'hombro_izq': [x,y], 'hombro_der': [x,y],
                              'cadera_izq': [x,y], 'cadera_der': [x,y]}
    """
    # Calculamos el centro de los hombros (promedio de izq y der)
    hombro_izq     = np.array(landmarks_np['hombro_izq'])
    hombro_der     = np.array(landmarks_np['hombro_der'])
    centro_hombros = (hombro_izq + hombro_der) / 2  # punto medio entre los 2 hombros

    # Centro de las caderas
    cadera_izq     = np.array(landmarks_np['cadera_izq'])
    cadera_der     = np.array(landmarks_np['cadera_der'])
    centro_caderas = (cadera_izq + cadera_der) / 2   # punto medio entre las 2 caderas

    # Vector tronco: apunta desde las caderas hacia los hombros
    vector_tronco = centro_hombros - centro_caderas

    # Vector vertical: apunta hacia arriba (en coordenadas de imagen, y decrece hacia arriba)
    vertical = np.array([0, -1])  # [x=0, y=-1] = dirección hacia arriba

    # Ángulo entre el tronco y la vertical usando producto punto
    # + 1e-6 previene división entre cero si el vector es muy pequeño
    coseno = np.dot(vector_tronco, vertical) / (
        np.linalg.norm(vector_tronco) * np.linalg.norm(vertical) + 1e-6
    )
    return round(math.degrees(math.acos(np.clip(coseno, -1, 1))), 1)


def detectar_anomalias_postura(imagen_rgb):
    """
    Detecta caídas, espalda encorvada y asimetría de hombros.
    Requiere cuerpo completo o al menos el tronco visible.
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]

    opts = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_POSE_FULL),
        running_mode=RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
    )

    with PoseLandmarker.create_from_options(opts) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.pose_landmarks:
            print("⚠️  No se detectó persona.")
            return imagen_anotada, []

        lms = resultado.pose_landmarks[0]  # lista plana de 33 NormalizedLandmark

        # Construimos un diccionario de coordenadas [x, y, visibility]
        # para todos los landmarks del cuerpo
        coords = {}
        for nombre, idx in POSE_LANDMARKS.items():
            lm = lms[idx]
            coords[nombre] = [lm.x, lm.y, lm.visibility]  # visibility es el tercer elemento

        # Dibuja el esqueleto base
        mp_drawing.draw_landmarks(
            imagen_anotada, lms,
            PoseLandmarksConnections.POSE_LANDMARKS,
            mp_drawing_styles.get_default_pose_landmarks_style(),
        )

        alertas = []  # lista de (mensaje, color) para las alertas detectadas

        # ── DETECCIÓN 1: CAÍDA ─────────────────────────────────────
        # Solo analizamos si los puntos de cadera y tobillo son visibles
        if coords['cadera_izq'][2] > 0.5 and coords['tobillo_izq'][2] > 0.5:
            # diff_y = distancia vertical entre tobillo y cadera
            # En posición normal: el tobillo está MUCHO más abajo (y mayor) que la cadera
            # En una caída: el tobillo y la cadera están casi al mismo nivel (diff_y pequeña)
            diff_y = coords['tobillo_izq'][1] - coords['cadera_izq'][1]

            angulo_tronco = calcular_angulo_tronco({k: v[:2] for k, v in coords.items()})
            # dict comprehension: {k: v[:2] ...} crea nuevo dict con solo x,y (sin visibility)

            if (diff_y < UMBRALES_POSTURA['caida_diferencia_y'] or
                    angulo_tronco > UMBRALES_POSTURA['caida_angulo_tronco']):
                alertas.append(('🚨 CAÍDA DETECTADA', (0, 0, 255)))
                # Dibuja un borde rojo grueso alrededor de toda la imagen
                cv2.rectangle(imagen_anotada, (5, 5), (ancho-5, alto-5), (0, 0, 255), 8)

        # ── DETECCIÓN 2: ESPALDA ENCORVADA ────────────────────────
        # Verificamos que los 3 puntos necesarios sean visibles
        # all([...]) devuelve True solo si TODOS los elementos son True
        if all(coords[p][2] > 0.5 for p in ['hombro_izq', 'cadera_izq', 'rodilla_izq']):
            angulo_espalda = calcular_angulo(
                coords['hombro_izq'][:2],
                coords['cadera_izq'][:2],
                coords['rodilla_izq'][:2]
            )
            if angulo_espalda < UMBRALES_POSTURA['espalda_angulo_min']:
                alertas.append((f'⚠️  ESPALDA ENCORVADA ({angulo_espalda}°)', (0, 165, 255)))

        # ── DETECCIÓN 3: ASIMETRÍA DE HOMBROS ─────────────────────
        if coords['hombro_izq'][2] > 0.5 and coords['hombro_der'][2] > 0.5:
            # Diferencia de altura entre hombros (convertida a píxeles)
            diff_hombros = abs(coords['hombro_izq'][1] - coords['hombro_der'][1]) * alto
            if diff_hombros > UMBRALES_POSTURA['hombros_nivel_max']:
                alertas.append((f'⚠️  HOMBROS ASIMÉTRICOS ({diff_hombros:.0f}px dif)', (0, 200, 200)))

        # Si no hay alertas, la postura es correcta
        if not alertas:
            alertas.append(('✅ POSTURA CORRECTA', (0, 200, 0)))

        # ── Mostrar alertas sobre la imagen ───────────────────────
        for i, (alerta, color) in enumerate(alertas):
            y_pos = 50 + i * 45  # posición vertical de cada alerta
            (tw, th), _ = cv2.getTextSize(alerta, cv2.FONT_HERSHEY_DUPLEX, 0.9, 2)
            cv2.rectangle(imagen_anotada, (10, y_pos-30), (tw+20, y_pos+10), (0, 0, 0), -1)
            cv2.putText(imagen_anotada, alerta, (15, y_pos),
                       cv2.FONT_HERSHEY_DUPLEX, 0.9, color, 2)

        print("=" * 55)
        print("  📊 REPORTE DE POSTURA")
        print("=" * 55)
        for alerta, _ in alertas:
            print(f"  {alerta}")
        print("=" * 55)

    return imagen_anotada, alertas


# ─── Widget ───────────────────────────────────────────────────────
uploader_postura = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🧍 Analizar postura')
btn_postura      = widgets.Button(description='Detectar Anomalías', button_style='danger')
salida_postura   = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen de cuerpo completo para análisis de postura:</b>'),
    uploader_postura, btn_postura, salida_postura
]))


def on_postura(_):
    with salida_postura:
        clear_output()
        if not uploader_postura.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_postura.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada, alertas = detectar_anomalias_postura(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Análisis de postura')


btn_postura.on_click(on_postura)
print("✅ Módulo 6 listo.")


---
# 🟡 MÓDULO 7 — YOLOv8-pose: Tecnología Alternativa

## ¿Qué es YOLO?

YOLO = **You Only Look Once** (Lo ves solo una vez).
Es un modelo de detección de objetos que en una sola pasada:
- Detecta MÚLTIPLES personas con bounding boxes
- Localiza 17 keypoints COCO por persona

## Keypoints COCO (estándar diferente a MediaPipe)

```
0=nariz,      1=ojo_izq,    2=ojo_der,    3=oreja_izq,  4=oreja_der
5=hombro_izq, 6=hombro_der, 7=codo_izq,   8=codo_der
9=muneca_izq, 10=muneca_der 11=cadera_izq, 12=cadera_der
13=rodilla_izq, 14=rodilla_der, 15=tobillo_izq, 16=tobillo_der
```

## ¿Cuándo usar YOLO en lugar de MediaPipe?

| Situación | Recomendación |
|---|---|
| 1 persona, análisis preciso | MediaPipe Full/Heavy |
| Múltiples personas | YOLOv8-pose |
| Tiempo real en CPU | MediaPipe Lite |
| También necesitas bounding box | YOLOv8-pose |
| También necesitas rostro/manos | MediaPipe Holistic |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 7 — YOLOv8-POSE                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

from ultralytics import YOLO
# La librería ultralytics contiene YOLOv8.
# YOLO es una clase que encapsula el modelo, la inferencia y el postprocesamiento.

# ─── Nombres de los 17 keypoints del estándar COCO ────────────────
# COCO = Common Objects in Context (dataset de referencia en visión artificial)
# El orden es FIJO y es el mismo en todos los modelos que usan el estándar COCO.
COCO_KEYPOINTS = [
    'nariz',       # 0  — nariz
    'ojo_izq',     # 1  — ojo izquierdo
    'ojo_der',     # 2  — ojo derecho
    'oreja_izq',   # 3  — oreja izquierda
    'oreja_der',   # 4  — oreja derecha
    'hombro_izq',  # 5  — hombro izquierdo
    'hombro_der',  # 6  — hombro derecho
    'codo_izq',    # 7  — codo izquierdo
    'codo_der',    # 8  — codo derecho
    'muneca_izq',  # 9  — muñeca izquierda
    'muneca_der',  # 10 — muñeca derecha
    'cadera_izq',  # 11 — cadera izquierda
    'cadera_der',  # 12 — cadera derecha
    'rodilla_izq', # 13 — rodilla izquierda
    'rodilla_der', # 14 — rodilla derecha
    'tobillo_izq', # 15 — tobillo izquierdo
    'tobillo_der', # 16 — tobillo derecho
]

# ─── Conexiones del esqueleto COCO ────────────────────────────────
# Pares de índices que se deben conectar con líneas para visualizar el esqueleto.
COCO_SKELETON = [
    (0, 1), (0, 2),      # nariz → ojos
    (1, 3), (2, 4),      # ojos → orejas
    (5, 6),              # hombros entre sí (línea de hombros)
    (5, 7), (7, 9),      # brazo izquierdo: hombro-codo-muñeca
    (6, 8), (8, 10),     # brazo derecho
    (5, 11), (6, 12),    # torso: hombros → caderas
    (11, 12),            # caderas entre sí (línea de caderas)
    (11, 13), (13, 15),  # pierna izquierda: cadera-rodilla-tobillo
    (12, 14), (14, 16),  # pierna derecha
]


def analizar_con_yolopose(imagen_rgb, conf_umbral=0.5):
    """
    Detecta personas y sus 17 keypoints COCO usando YOLOv8-pose.
    Soporta múltiples personas simultáneamente.

    Parámetros:
        imagen_rgb   → array NumPy RGB
        conf_umbral  → confianza mínima para considerar una detección (0.0-1.0)

    Retorna:
        imagen_anotada → imagen con bounding boxes y keypoints
        tabla          → DataFrame con keypoints de todas las personas
    """
    imagen_anotada = imagen_rgb.copy()

    # ── Cargar el modelo YOLOv8-pose ──────────────────────────────
    # 'yolov8n-pose.pt' = variante 'nano': el modelo más pequeño y rápido.
    # Variantes disponibles (de menor a mayor): n, s, m, l, x
    # Si el archivo no existe, se descarga automáticamente (~6MB).
    print("⬇️  Cargando modelo YOLOv8n-pose...")
    modelo = YOLO('yolov8n-pose.pt')
    print("✅ Modelo listo.")

    # ── Ejecutar inferencia ───────────────────────────────────────
    # model.predict() ejecuta la detección.
    # Devuelve una LISTA de resultados (uno por imagen en el batch).
    # Como enviamos 1 imagen, resultados[0] es nuestro resultado.
    resultados = modelo.predict(
        source=imagen_rgb,   # array NumPy (también acepta rutas, URLs, videos)
        conf=conf_umbral,    # umbral de confianza mínima
        verbose=False        # False = silencia la barra de progreso
    )

    res = resultados[0]  # resultado de la primera (única) imagen

    # ── Verificar detecciones ─────────────────────────────────────
    # res.keypoints.data tiene forma (N_personas, 17, 3)
    # donde 3 = [x_píxeles, y_píxeles, confianza_del_punto]
    if res.keypoints is None or len(res.keypoints.data) == 0:
        print("⚠️  No se detectaron personas.")
        return imagen_anotada, pd.DataFrame()

    n_personas = len(res.keypoints.data)
    print(f"✅ {n_personas} persona(s) detectada(s).")

    # Colores diferentes para cada persona (hasta 5)
    COLORES = [(255, 100, 0), (0, 200, 100), (100, 0, 255), (255, 0, 150), (0, 150, 255)]

    todas_filas = []

    for idx_persona in range(n_personas):
        color = COLORES[idx_persona % len(COLORES)]

        # ── Dibujar bounding box ───────────────────────────────────
        # res.boxes.xyxy[i] → tensor [x1, y1, x2, y2] de la caja i
        # .cpu() mueve el tensor de GPU a CPU (necesario para convertir a NumPy)
        # .numpy() convierte a array NumPy
        # .astype(int) convierte a enteros (cv2 requiere coordenadas enteras)
        if res.boxes is not None and idx_persona < len(res.boxes):
            box = res.boxes.xyxy[idx_persona].cpu().numpy().astype(int)
            cv2.rectangle(imagen_anotada, (box[0], box[1]), (box[2], box[3]), color, 2)
            confianza_box = float(res.boxes.conf[idx_persona])
            cv2.putText(imagen_anotada, f"Persona {idx_persona+1} ({confianza_box:.0%})",
                       (box[0], box[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # ── Extraer keypoints ──────────────────────────────────────
        # kpts tiene forma (17, 3): [x, y, confianza] por cada keypoint
        kpts = res.keypoints.data[idx_persona].cpu().numpy()

        # ── Dibujar conexiones del esqueleto ──────────────────────
        for (i1, i2) in COCO_SKELETON:
            x1, y1, c1 = kpts[i1]
            x2, y2, c2 = kpts[i2]
            # Solo dibujamos si AMBOS puntos de la conexión son confiables (> 0.5)
            if c1 > 0.5 and c2 > 0.5:
                cv2.line(imagen_anotada, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)

        # ── Dibujar puntos individuales ────────────────────────────
        for idx_kpt, (x, y, conf) in enumerate(kpts):
            if conf > 0.5:
                px, py = int(x), int(y)
                cv2.circle(imagen_anotada, (px, py), 5, color, -1)          # punto coloreado
                cv2.circle(imagen_anotada, (px, py), 5, (255, 255, 255), 1) # borde blanco
                cv2.putText(imagen_anotada, COCO_KEYPOINTS[idx_kpt][:5],
                           (px+6, py-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 200), 1)

            todas_filas.append({
                'Persona': idx_persona + 1,
                'Keypoint': COCO_KEYPOINTS[idx_kpt],
                'Índice COCO': idx_kpt,
                'x (px)': round(float(x), 1),
                'y (px)': round(float(y), 1),
                'Confianza': round(float(conf), 3),
            })

    return imagen_anotada, pd.DataFrame(todas_filas)


# ─── Widget ───────────────────────────────────────────────────────
uploader_yolo = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🟡 YOLO imagen')
btn_yolo      = widgets.Button(description='Analizar con YOLO', button_style='warning')
salida_yolo   = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen (puede tener múltiples personas):</b>'),
    uploader_yolo, btn_yolo, salida_yolo
]))


def on_yolo(_):
    with salida_yolo:
        clear_output()
        if not uploader_yolo.value:
            print("⚠️  Primero sube una imagen."); return
        valor    = uploader_yolo.value
        archivo  = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb  = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada, tabla = analizar_con_yolopose(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'YOLOv8-pose (17 keypoints COCO)')
        if not tabla.empty:
            print("\n📊 Keypoints con confianza > 0.5:")
            display(tabla[tabla['Confianza'] > 0.5])


btn_yolo.on_click(on_yolo)
print("✅ Módulo 7 listo: analizar_con_yolopose()")


---
# 🎓 MÓDULO 8 — Resumen, Ejercicios y Próximos Pasos

## ¿Qué aprendiste?

| Módulo | Habilidad adquirida |
|---|---|
| 0 | Instalar librerías, importar módulos, crear funciones utilitarias |
| 1 | Face Mesh: 468 landmarks, EAR/MAR, somnolencia |
| 2 | Hand Landmarks: 21 puntos, esqueleto de la mano |
| 3 | Body Pose: 33 puntos, ángulos articulares biomecánicos |
| 4 | Holistic: 543 landmarks simultáneos, 3 detectores en 1 llamada |
| 5 | Detección de gestos con lógica geométrica + ML |
| 6 | Caídas y posturas incorrectas con análisis geométrico |
| 7 | YOLOv8-pose: 17 keypoints COCO, multi-persona |

---

## 🚀 Ejercicios propuestos por nivel

### 🟢 Nivel Básico

1. **Contador de dedos:** Modifica `clasificar_gesto()` para que cuente los dedos
   extendidos y los muestre como texto grande en la imagen.

2. **Alerta de somnolencia:** Usa `calcular_ear()` en un video. Si EAR < 0.20
   por más de 30 frames consecutivos, imprime `'⚠️ ATENCIÓN: conductor somnoliento'`.

3. **Mapa de calor de landmarks:** Para una imagen con muchas personas (imagen grupal),
   acumula las posiciones de la nariz de cada persona y visualízalas con un scatter plot.

### 🟡 Nivel Intermedio

4. **Contador de flexiones:** En un video de ejercicio, cuenta cuántas veces el ángulo
   del codo pasa por debajo de 90° y vuelve a subir por encima de 150°.

5. **Reconocimiento de letras ASL:** Usa los 63 valores (21 landmarks × 3 coords)
   de la mano como features y entrena un clasificador scikit-learn con las
   26 letras del alfabeto de señas americano.

6. **Trayectoria de movimiento:** Dibuja la trayectoria de la muñeca durante
   un video (cola de 30 frames) con un degradado de color para mostrar la dirección.

### 🔴 Nivel Avanzado

7. **Análisis de sentadilla (squat):** Detecta si la sentadilla es correcta:
   - Rodillas > 90° en el punto más bajo
   - Espalda < 45° de inclinación
   - Rodillas alineadas con los pies

8. **Multi-persona con ID de tracking:** Usa YOLOv8-pose para detectar múltiples
   personas y asigna un ID persistente entre frames usando la distancia entre
   centroides en frames consecutivos.

9. **Dataset para ML:** Graba 10 videos de 5 posturas diferentes. Extrae los
   33 landmarks + 10 ángulos de cada frame como fila en un CSV.
   Entrena un Random Forest para clasificar las posturas en tiempo real.

---

## 📚 Recursos para seguir aprendiendo

```
📖 Documentación oficial:
   MediaPipe → https://ai.google.dev/edge/mediapipe/solutions/guide
   YOLOv8    → https://docs.ultralytics.com
   OpenCV    → https://docs.opencv.org

📄 Papers científicos clave:
   BlazePose (MediaPipe) → arxiv.org/abs/2006.10204
   OpenPose             → arxiv.org/abs/1812.08008

🗃️ Datasets para entrenar modelos propios:
   COCO Keypoints  → cocodataset.org
   Human3.6M       → vision.imar.ro/human3.6m
   MPII Human Pose → human-pose.mpi-inf.mpg.de
```

---

## 🏗️ Arquitectura para proyectos en producción

```
ENTRADA
  ├── Cámara IP (RTSP)   → cv2.VideoCapture('rtsp://...')
  ├── Archivo de video   → cv2.VideoCapture('video.mp4')
  └── Batch de imágenes  → for archivo in lista_de_archivos:

PROCESAMIENTO
  ├── 1 persona en tiempo real → MediaPipe Pose Lite
  ├── 1 persona alta precisión → MediaPipe Pose Heavy
  └── Múltiples personas       → YOLOv8-pose

LÓGICA DE NEGOCIO
  ├── Caídas   → alerta SMS/push notification
  ├── Deportes → métricas y feedback en tiempo real
  ├── Control  → gestos como comandos de interfaz
  └── Datos    → exportar a CSV, base de datos

SALIDA
  ├── Video anotado   → cv2.VideoWriter()
  ├── Dashboard web   → Streamlit / Flask
  └── Datos           → pandas → CSV / SQLite
```

---

> 🎯 **Consejo final:** La mejor forma de aprender visión artificial es
> **experimentar**. Toma cualquier función de este notebook, modifica los
> umbrales, cambia los colores, agrega un nuevo landmark. Cada modificación
> que funciona es un concepto que internalizaste para siempre.

**¡Felicidades por completar el notebook! 🎓**
